<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/%E5%B7%A6%E5%8F%B3%E5%85%A9%E5%81%B4%E6%B7%B7%E5%90%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gc
import logging
import os
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import yfinance as yf
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

try:
    from IPython.display import display
except ImportError:
    def display(df):
        print(df.to_string())

CACHE_DIR = "./chip_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.58  # 門檻：捕捉發動訊號
PORTFOLIO_BASE_CAPITAL = 1_000_000
TRANSACTION_COST_PCT = 0.004

http_session = requests.Session()
retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
http_session.mount("https://", HTTPAdapter(max_retries=retries))

# ==============================================================================
# 0. 股票資料池定義 (完整保持 160 隻標的)
# ==============================================================================
stock_dict = {
    "0050.TW": "元大台灣50", "0056.TW": "元大高股息", "00878.TW": "國泰永續高股息", "00770.TW": "國泰北美科技", "00981A.TW": "統一台股增長主動式", "SPCX": "SPACs ETF", "SOXX": "iShares半導體ETF", "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果", "GOOG": "Google / Alphabet", "META": "Meta", "MSFT": "Microsoft 微軟", "NVDA": "NVIDIA 輝達", "TSM": "台積電 ADR", "TSLA": "Tesla 特斯拉", "ENTG": "Entegris 英特格", "SMR": "NuScale Power 小型核反應爐", "BE": "Bloom Energy 燃料電池", "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾", "AMAT": "Applied Materials 應用材料", "LRCX": "Lam Research 柯林研發", "KLAC": "KLA 科磊", "AMD": "AMD 超微", "AVGO": "Broadcom 博通", "QCOM": "Qualcomm 高通", "INTC": "Intel 英特爾", "MU": "Micron 鎂光", "TXN": "Texas Instruments 德州儀器", "ARM": "ARM 晶心/安謀", "MRVL": "Marvell 邁威爾", "ADI": "Analog Devices 亞德諾", "MPWR": "Monolithic Power 芯源系統", "ON": "ON Semiconductor 安森美", "SWKS": "Skyworks 思佳訊", "QRVO": "Qorvo 威訊", "TER": "Teradyne 泰瑞達", "MKSI": "MKS Instruments",
    "PANW": "Palo Alto Networks", "CRWD": "CrowdStrike", "FTNT": "Fortinet", "NET": "Cloudflare", "ZS": "Zscaler", "OKTA": "Okta", "S": "SentinelOne", "GEN": "Gen Digital", "RPD": "Rapid7", "CBRS": "CyberArk",
    "2471.TW": "資通", "2480.TW": "敦陽科", "3029.TW": "零壹", "6214.TW": "精誠", "3130.TW": "一零四", "2427.TW": "三商電", "3027.TW": "盛達", "5203.TW": "訊連", "5471.TW": "松翰", "5410.TW": "國統", "6183.TW": "關貿", "6203.TWO": "海韻電", "6210.TWO": "慶生", "6593.TWO": "台灣銘板", "6689.TW": "伊雲谷", "6690.TWO": "安碁資訊", "6752.TWO": "睿嘉", "6763.TWO": "綠界科技", "6865.TWO": "偉康科技", "6874.TWO": "倍力", "6928.TW": "全達",
    "2382.TW": "廣達", "3231.TW": "緯創", "6669.TW": "緯穎", "2317.TW": "鴻海", "2356.TW": "英業達", "2324.TW": "仁寶", "2376.TW": "技嘉", "3706.TW": "神達", "2377.TW": "微 MSI", "2357.TW": "華碩", "4938.TW": "和碩", "3005.TW": "神基", "2353.TW": "宏碁",
    "2330.TW": "台積電", "2303.TW": "聯電", "2454.TW": "聯發科", "3034.TW": "聯詠", "3661.TW": "世芯-KY", "3443.TW": "創意", "4961.TW": "天鈺", "6415.TW": "矽力-KY", "6531.TW": "愛普*", "3035.TW": "智原", "6643.TWO": "M31", "4966.TWO": "譜瑞-KY", "5269.TW": "祥碩", "6104.TWO": "創唯", "6756.TW": "威鋒電子",
    "2342.TW": "茂矽", "6770.TW": "力積電", "3707.TWO": "漢磊", "3016.TW": "嘉晶", "3711.TW": "日月光投控", "2449.TW": "京元電子", "6257.TW": "矽格", "3264.TWO": "欣銓", "6239.TW": "力成", "2329.TW": "華泰", "2441.TW": "超豐",
    "3131.TWO": "弘塑", "3583.TW": "辛耘", "6187.TWO": "萬潤", "2467.TW": "志聖", "8027.TWO": "钛昇", "5434.TW": "崇越", "3010.TW": "華立", "1560.TW": "中砂", "3680.TWO": "家登", "5234.TW": "達興材料", "4749.TWO": "新應材", "8028.TW": "昇陽半導體", "6515.TW": "穎崴", "6683.TWO": "雍智科技", "6510.TWO": "精測", "6223.TWO": "旺矽",
    "2404.TW": "漢唐", "1773.TW": "勝一", "6196.TW": "帆宣", "6139.TW": "亞翔", "6613.TWO": "朋億*", "4755.TW": "三福化", "4768.TW": "晶呈科技", "3563.TW": "牧德", "3167.TW": "大量", "6438.TW": "迅得", "1595.TWO": "川寶",
    "6147.TWO": "頎邦", "8150.TW": "南茂", "6552.TW": "易華電", "5536.TWO": "聖暉*", "3644.TWO": "凌嘉科", "7769.TW": "鴻勁",
    "2344.TW": "華邦電", "2408.TW": "南亞科", "2337.TW": "旺宏", "3006.TW": "晶豪科", "3260.TWO": "威剛", "2451.TW": "創見", "4967.TW": "十銓", "8271.TW": "宇瞻", "5289.TWO": "宜晶", "8299.TWO": "群聯", "5351.TWO": "鈺創",
    "2308.TW": "台達電", "2301.TW": "光寶科", "6282.TW": "康舒", "6412.TW": "群電", "3665.TW": "貿聯-KY", "3017.TW": "奇鋐", "3324.TWO": "雙鴻", "3653.TW": "健策", "2421.TW": "建準", "8996.TW": "高力", "3483.TWO": "力致", "6230.TW": "尼得科超眾", "3013.TW": "晟銘電", "6805.TW": "富世達", "8210.TW": "勤誠", "6117.TW": "迎廣", "6235.TW": "華孚", "2354.TW": "鴻準", "3376.TW": "新日興", "3548.TWO": "兆利", "5243.TW": "乙盛-KY", "6715.TW": "嘉基", "3533.TW": "嘉澤", "3217.TWO": "優群", "3023.TW": "信邦", "2392.TW": "正崴", "3689.TWO": "湧德", "3357.TWO": "臺慶科", "6862.TW": "三集瑞-KY", "6821.TWO": "聯寶", "3207.TWO": "耀勝", "6197.TW": "佳必股", "8103.TW": "瀚荃", "3526.TWO": "凡甲", "3605.TW": "宏致", "2059.TW": "川湖", "6584.TWO": "南俊國際",
    "2327.TW": "國巨", "2492.TW": "華新科", "2375.TW": "凱美", "2478.TW": "大毅", "3026.TW": "禾伸堂", "3090.TW": "日電貿", "6173.TWO": "信昌電", "6155.TW": "鈞寶", "6175.TWO": "立敦", "5328.TWO": "華容", "3236.TWO": "千如", "8043.TWO": "蜜望實",
    "3037.TW": "欣興", "8046.TW": "南電", "3189.TW": "景碩", "4958.TW": "臻鼎-KY", "2368.TW": "金像電", "3044.TW": "健鼎", "2313.TW": "華通", "8155.TWO": "博智", "2383.TW": "台光電", "6274.TWO": "台燿", "6213.TW": "聯茂", "1717.TW": "長興", "1815.TWO": "富喬", "1802.TW": "台玻", "5340.TWO": "建榮", "5475.TWO": "德宏", "3305.TW": "昇貿", "3631.TWO": "晟楠", "8358.TWO": "金居", "8021.TW": "尖點", "6672.TW": "騰輝電子-KY",
    "2345.TW": "智邦", "5388.TW": "中磊", "3558.TWO": "神準", "3704.TW": "合勤控", "4906.TW": "正文", "4979.TWO": "華星光", "6442.TW": "光聖", "4908.TWO": "前鼎", "3163.TWO": "波若威", "3450.TW": "聯鈞", "6426.TW": "統新", "4977.TW": "眾達-KY", "6530.TWO": "創威", "3363.TWO": "上詮", "3234.TWO": "光環", "4903.TWO": "聯光通", "3081.TWO": "聯亞", "4991.TWO": "環宇-KY", "4971.TWO": "IET-KY", "6588.TWO": "東典光電", "3491.TWO": "昇達科", "2314.TW": "台揚", "6285.TW": "啟碁", "3105.TWO": "穩懋", "2455.TW": "全新", "3138.TW": "耀登", "2419.TW": "仲琦",
    "2395.TW": "研華", "6166.TW": "凌華", "8050.TWO": "廣積", "3556.TWO": "禾瑞亞", "2414.TW": "精技", "6414.TW": "樺漢", "3022.TW": "威強電", "2397.TW": "友通", "5314.TWO": "世紀",
    "6781.TW": "AES-KY", "3211.TWO": "順達", "6121.TWO": "新普", "3323.TWO": "加百裕", "3625.TWO": "西勝", "8038.TWO": "長園科", "4931.TWO": "新盛力",
    "1519.TW": "華城", "1513.TW": "中興電", "1514.TW": "亞力", "1503.TW": "士電", "1609.TW": "大亞", "1605.TW": "華新", "1608.TW": "華榮", "6869.TW": "雲豹能源", "2049.TW": "上銀", "4576.TW": "大銀微系統", "4585.TW": "達明", "2359.TW": "所羅門", "6188.TWO": "廣明", "8374.TW": "羅昇", "5443.TWO": "均豪", "6640.TWO": "均華", "2464.TW": "盟立", "6215.TW": "和椿", "4562.TW": "穎漢", "1590.TW": "亞德客-KY", "1504.TW": "東元",
    "3481.TW": "群創", "2409.TW": "友達", "3008.TW": "大立光", "4915.TW": "先進光", "5288.TW": "匯鑽科", "2393.TW": "億光",
    "2201.TW": "裕隆", "2204.TW": "中華", "2206.TW": "三陽工業", "1536.TW": "和大", "2231.TW": "聯嘉", "3552.TWO": "同致", "6279.TWO": "胡連",
    "2603.TW": "長榮", "2609.TW": "陽明", "2615.TW": "萬海", "2605.TW": "新興", "2606.TW": "裕民", "2612.TW": "中航", "2617.TW": "台航", "2637.TW": "慧洋-KY", "2641.TWO": "正德", "5608.TW": "四維航", "2610.TW": "華航", "2618.TW": "長榮航", "2630.TW": "亞航", "5603.TWO": "陸海", "2607.TW": "勞運", "2608.TW": "嘉里大榮", "2611.TW": "志信", "2613.TW": "中櫃", "2636.TW": "台驊投控", "2642.TW": "宅配通", "2633.TW": "台灣高鐵", "5607.TW": "遠雄港", "5609.TWO": "中菲行", "8367.TW": "建新國際",
    "2892.TW": "第一金", "5880.TW": "合庫金", "1210.TW": "大成", "1215.TW": "卜蜂", "1216.TW": "統一", "2912.TW": "統一超", "5903.TWO": "全家", "1303.TW": "南亞", "2465.TW": "麗臺", "8163.TW": "達方", "3042.TW": "晶技", "8182.TWO": "加高", "3229.TW": "泰藝", "3308.TW": "聯傑", "6284.TWO": "佳邦", "2484.TW": "希華", "8088.TWO": "華信科",
}

download_cache = {}

def classify_industry(code, name):
    code_str = str(code).upper()
    name_str = str(name)
    if code_str in ["JNJ"]:
        return "美股醫療保健/生技製藥"
    elif code_str in ["GEN"]:
        return "美股科技/軟體資安"
    elif code_str.startswith("00") or "ETF" in name_str or "增長" in name_str or code_str in ["SPCX", "SOXX", "SMH"]:
        return "ETF/大盤指數"
    elif (code_str.startswith("28") or code_str.startswith("58")) and ("金" in name_str or "銀" in name_str or "保" in name_str):
        return "金融/保險"
    elif code_str.startswith("26") or code_str.startswith("56") or "航" in name_str or "海" in name_str or "宅配" in name_str or "高鐵" in name_str or "大榮" in name_str or "港" in name_str:
        return "航運/物流"
    elif code_str.startswith("12") or code_str.startswith("29") or code_str.startswith("59") or "統一超" in name_str or "全家" in name_str or "食品" in name_str:
        return "食品/民生消費"
    elif code_str in ["AAPL", "META", "GOOG", "MSFT", "NVDA", "TSM", "ASML", "AMAT", "LRCX", "KLAC", "AMD", "AVGO", "QCOM", "INTC", "MU", "TXN", "ARM", "MRVL", "ADI", "MPWR", "ON", "SWKS", "QRVO", "TER", "MKSI", "PANW", "CRWD", "FTNT", "NET", "ZS", "OKTA", "S", "RPD", "CBRS", "ENTG", "SMR", "BE"]:
        return "美股科技/半導體"
    else:
        return "台股電子/半導體/供應鏈"

def fetch_twse_foreign_bulk(date_str):
    cache_file = os.path.join(CACHE_DIR, f"twse_{date_str}.json")
    if os.path.exists(cache_file):
        try:
            return pd.read_json(cache_file)
        except Exception:
            pass
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={date_str}&selectType=ALLBUT0999&response=json"
    try:
        time.sleep(0.15)
        res = http_session.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5, verify=False)
        if res.status_code == 200:
            js = res.json()
            if js.get("stat") == "OK":
                df = pd.DataFrame(js["data"], columns=js["fields"])
                df.to_json(cache_file)
                return df
    except Exception:
        pass
    return None

def fetch_tpex_foreign_bulk(date_str_slash):
    date_clean = date_str_slash.replace("/", "")
    cache_file = os.path.join(CACHE_DIR, f"tpex_{date_clean}.json")
    if os.path.exists(cache_file):
        try:
            return pd.read_json(cache_file)
        except Exception:
            pass
    url = f"https://www.tpex.org.tw/www/zh-tw/insti/qfiiStat?type=Daily&date={date_str_slash}&searchType=buy&id=&response=json"
    try:
        time.sleep(0.15)
        res = http_session.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5, verify=False)
        if res.status_code == 200:
            js = res.json()
            if "tables" in js and len(js["tables"]) > 0:
                t = js["tables"][0]
                df = pd.DataFrame(t["data"], columns=t["fields"])
                df.to_json(cache_file)
                return df
    except Exception:
        pass
    return None

def sanitize_df(df):
    if df is None or df.empty:
        return None
    d = df.copy()
    if d.index.tz is not None:
        d.index = d.index.tz_localize(None)
    if isinstance(d.columns, pd.MultiIndex):
        for level in range(d.columns.nlevels):
            col_names = [str(c).strip().title() for c in d.columns.get_level_values(level)]
            if "Close" in col_names:
                d.columns = d.columns.get_level_values(level)
                break
        else:
            d.columns = d.columns.get_level_values(-1)
    col_map = {c: str(c).strip().title().replace("Adj Close", "Close") for c in d.columns}
    d = d.rename(columns=col_map)
    needed = ["Open", "Close", "High", "Low", "Volume"]
    return d[needed].dropna(subset=["Close"]) if all(k in d.columns for k in needed) else None

def compute_market_features(market_df):
    m = sanitize_df(market_df)
    if m is None:
        return pd.DataFrame()
    m_returns = m["Close"].pct_change()
    return pd.DataFrame({
        "Market_Vol_20": m_returns.rolling(20).std() * np.sqrt(252),
        "Market_Ret_20": m["Close"].pct_change(20),
        "Market_MA_Dist": (m["Close"] - m["Close"].rolling(20).mean()) / (m["Close"].rolling(20).mean() + 1e-6),
        "Market_Close": m["Close"],
    }, index=m.index)

def get_market_data():
    try:
        tw_feats = compute_market_features(yf.download("^TWII", period="3y", progress=False, auto_adjust=True))
    except Exception:
        tw_feats = compute_market_features(yf.download("0050.TW", period="3y", progress=False, auto_adjust=True))
    try:
        us_feats = compute_market_features(yf.download("^GSPC", period="3y", progress=False, auto_adjust=True))
    except Exception:
        us_feats = compute_market_features(yf.download("SPY", period="3y", progress=False, auto_adjust=True))
    return tw_feats, us_feats

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def download_stock_with_auto_suffix(ticker, period="1y"):
    if ticker in download_cache:
        return download_cache[ticker]
    pure_t = ticker.split(".")[0]
    candidates = [f"{pure_t}.TWO", f"{pure_t}.TW", ticker] if pure_t.isdigit() and len(pure_t) >= 4 else [ticker]
    candidates = list(dict.fromkeys(candidates))
    for cand in candidates:
        try:
            df = yf.download(cand, period=period, progress=False, auto_adjust=True)
            cleaned_df = sanitize_df(df)
            if cleaned_df is not None and len(cleaned_df) > 50:
                download_cache[ticker] = (cleaned_df, cand)
                return cleaned_df, cand
        except Exception:
            continue
    return None, ticker

def compute_features(df, market_feats, is_tw_stock=False, ticker=None, twse_cache=None, tpex_cache=None):
    d = df.copy()
    if d is None or len(d) < 200:
        return None, []

    if twse_cache is None: twse_cache = {}
    if tpex_cache is None: tpex_cache = {}

    x = np.arange(5)
    x_dev = x - x.mean()
    x_var = (x_dev**2).sum()
    close_vals = d["Close"].values
    if len(close_vals) >= 5:
        shape = (len(close_vals) - 5 + 1, 5)
        strides = (close_vals.strides[0], close_vals.strides[0])
        windows = np.lib.stride_tricks.as_strided(close_vals, shape=shape, strides=strides)
        y_dev = windows - windows.mean(axis=1, keepdims=True)
        slopes = (y_dev * x_dev).sum(axis=1) / x_var
        pad_slopes = np.concatenate([np.repeat(np.nan, 4), slopes])
        d["Close_Slope"] = pad_slopes / (d["Close"].values + 1e-6)
    else:
        d["Close_Slope"] = 0.0

    tr = pd.concat([
        d["High"] - d["Low"],
        (d["High"] - d["Close"].shift(1)).abs(),
        (d["Low"] - d["Close"].shift(1)).abs()
    ], axis=1).max(axis=1)
    d["ATR_14"] = tr.rolling(14).mean()
    d["NATR"] = d["ATR_14"] / (d["Close"] + 1e-6)
    d["Hist_Vol_20"] = d["Close"].pct_change().rolling(20).std() * np.sqrt(252)

    ma5 = d["Close"].rolling(5).mean()
    ma20 = d["Close"].rolling(20).mean()
    ma60 = d["Close"].rolling(60).mean()
    d["MA60"] = ma60
    d["MA_Bullish_Align"] = ((d["Close"] > ma5) & (ma5 > ma20) & (ma20 > ma60)).astype(float)
    std20 = d["Close"].rolling(20).std()
    d["BB_Bandwidth"] = (4 * std20) / (ma20 + 1e-6)
    d["BB_Squeeze"] = d["BB_Bandwidth"] / (d["BB_Bandwidth"].rolling(60).mean() + 1e-6)
    d["BIAS_5"] = (d["Close"] - ma5) / (ma5 + 1e-6)

    ema12, ema26 = d["Close"].ewm(span=12).mean(), d["Close"].ewm(span=26).mean()
    d["MACD_Hist"] = (ema12 - ema26 - (ema12 - ema26).ewm(span=9).mean()) / (d["Close"] + 1e-6)
    d["MACD_Hist_Slope"] = d["MACD_Hist"].diff(3)
    d["RSI_14"] = compute_rsi(d["Close"], 14) / 100.0
    d["RSI_Slope"] = d["RSI_14"].diff(3)

    mfv = (((d["Close"] - d["Low"]) - (d["High"] - d["Close"])) / (d["High"] - d["Low"] + 1e-6)) * d["Volume"]
    d["CMF_20"] = (mfv.rolling(20).sum() / (d["Volume"].rolling(20).sum() + 1e-6)).fillna(0)

    d["Volume_Explosion"] = np.clip(d["Volume"] / (d["Volume"].rolling(5).mean() + 1e-6), 0, 10)
    d["Turnover_Rate"] = np.clip(d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6), 0, 10)
    d["Body_Ratio"] = (d["Close"] - d["Open"]).abs() / (d["High"] - d["Low"] + 1e-6)

    d["Vol_MA5"] = d["Volume"].rolling(5).mean()
    d["Turnover_MA5"] = (d["Volume"] * d["Close"]).rolling(5).mean()

    d["Foreign_Net_Vol_Ratio"] = 0.0
    if is_tw_stock and ticker:
        pure_ticker = ticker.split(".")[0]
        f_dict = {}
        cache_source = twse_cache if ticker.endswith(".TW") else tpex_cache
        for dt, raw_dict in cache_source.items():
            if pure_ticker in raw_dict:
                f_dict[dt] = raw_dict[pure_ticker]
        if f_dict:
            f_series = pd.Series(f_dict)
            d["Foreign_Net_Vol_Ratio"] = f_series.reindex(d.index).fillna(0.0) / (d["Volume"] + 1e-6)

    d["Foreign_Net_MA5"] = d["Foreign_Net_Vol_Ratio"].rolling(5).mean().fillna(0)
    is_buy = (d["Foreign_Net_Vol_Ratio"] > 0).astype(int)
    is_not_buy = (d["Foreign_Net_Vol_Ratio"] <= 0).astype(int)
    d["Foreign_Buy_Streak"] = is_buy.groupby(is_not_buy.cumsum()).cumsum()

    d["Alpha_5d"] = d["Close"].pct_change(5) - market_feats["Market_Close"].pct_change(5)
    m_re = market_feats.reindex(d.index).ffill()
    ret_stock = d["Close"].pct_change()
    ret_mkt = m_re["Market_Close"].pct_change()
    cov_sm = ret_stock.rolling(20).cov(ret_mkt)
    var_m = ret_mkt.rolling(20).var()
    d["Rolling_Beta"] = (cov_sm / (var_m + 1e-6)).fillna(1.0)
    d["Market_Vol_20"], d["Market_Ret_20"], d["Market_MA_Dist"] = m_re["Market_Vol_20"], m_re["Market_Ret_20"], m_re["Market_MA_Dist"]

    liquidity_ok = d["Turnover_MA5"] >= (30_000_000 if is_tw_stock else 2_000_000)
    right_side_trend = (d["Close"] > ma60) & ((ma20 - ma20.rolling(5).mean()).abs() / ma20 <= 0.04)
    left_side_reversal = (
        (d["Close"] <= ma60 * 1.08) &
        (
            ((d["RSI_14"] < 0.45) & (d["RSI_Slope"] > 0)) |
            (d["MACD_Hist_Slope"] > 0) |
            ((d["Volume_Explosion"] > 1.2) & (d["Body_Ratio"] > 0.4) & (d["Close"] > d["Open"]))
        )
    )

    d["Filter_Pass"] = liquidity_ok & (right_side_trend | left_side_reversal)

    upper_barrier = d["Close"] + (d["ATR_14"] * 2.5)
    lower_barrier = d["Close"] - (d["ATR_14"] * 1.5)
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=10)
    f_max = d["High"].shift(-1).rolling(indexer).max()
    f_min = d["Low"].shift(-1).rolling(indexer).min()
    d["Target"] = ((f_max >= upper_barrier) & (f_min > lower_barrier)).astype(float)
    d.iloc[-10:, d.columns.get_loc("Target")] = np.nan

    f_cols = [
        "Close_Slope", "NATR", "Hist_Vol_20", "BB_Bandwidth", "BB_Squeeze", "BIAS_5",
        "MA_Bullish_Align", "Volume_Explosion", "Turnover_Rate", "Body_Ratio",
        "RSI_14", "RSI_Slope", "MACD_Hist", "MACD_Hist_Slope", "CMF_20",
        "Alpha_5d", "Rolling_Beta", "Market_Vol_20", "Market_Ret_20", "Market_MA_Dist",
        "Foreign_Net_Vol_Ratio", "Foreign_Net_MA5", "Foreign_Buy_Streak"
    ]

    d[f_cols] = d[f_cols].ffill(limit=1)
    return d, f_cols

if __name__ == "__main__":
    twse_foreign_cache, tpex_foreign_cache = {}, {}
    print("【系統初始化】下載雙市場大盤數據與籌碼庫...")
    tw_market_feats, us_market_feats = get_market_data()

    dummy_df, _ = download_stock_with_auto_suffix("2330.TW", period="1y")
    recent_dates = dummy_df.index[-60:] if dummy_df is not None else []

    for r_date in recent_dates:
        d_str, d_slash = r_date.strftime("%Y%m%d"), r_date.strftime("%Y/%m/%d")
        df_twse = fetch_twse_foreign_bulk(d_str)
        df_tpex = fetch_tpex_foreign_bulk(d_slash)
        if df_twse is not None and not df_twse.empty:
            twse_dict = {}
            for _, row in df_twse.iterrows():
                try:
                    code = str(row.iloc[0]).strip()
                    val_str = str(row.iloc[4]).replace(",", "").replace("-", "").strip()
                    if val_str.isdigit():
                        twse_dict[code] = float(str(row.iloc[4]).replace(",", ""))
                except Exception:
                    continue
            twse_foreign_cache[r_date] = twse_dict

        if df_tpex is not None and not df_tpex.empty:
            tpex_dict = {}
            for _, row in df_tpex.iterrows():
                try:
                    code = str(row.iloc[0]).strip()
                    val_str = str(row.iloc[4]).replace(",", "").replace("-", "").strip()
                    if val_str.isdigit():
                        tpex_dict[code] = float(str(row.iloc[4]).replace(",", ""))
                except Exception:
                    continue
            tpex_foreign_cache[r_date] = tpex_dict

    all_dfs, tickers = [], list(stock_dict.keys())
    print("【資料建構】載入與計算全套因子面板資料 (支援左右側混血策略)...")
    for t in tickers:
        try:
            df, corrected_ticker = download_stock_with_auto_suffix(t, period="1y")
            if df is not None:
                is_tw = corrected_ticker.endswith(".TW") or corrected_ticker.endswith(".TWO")
                df_feat, f_cols = compute_features(df, tw_market_feats if is_tw else us_market_feats, is_tw, corrected_ticker, twse_foreign_cache, tpex_foreign_cache)
                if df_feat is not None:
                    df_feat["Ticker"] = corrected_ticker
                    df_feat["Stock_Name"] = stock_dict.get(t, stock_dict.get(corrected_ticker, "未知標的"))
                    all_dfs.append(df_feat)
        except Exception:
            continue

    if not all_dfs:
        print("❌ 無法載入任何有效的股票資料，流程終止。")
        exit()

    panel_df = pd.concat(all_dfs).sort_index()
    unique_dates = panel_df.index.unique().sort_values()
    cutoff_date = unique_dates[-6]
    panel_clean = panel_df.dropna(subset=f_cols)

    hist_filtered = panel_clean[
        (panel_clean.index <= cutoff_date) &
        panel_clean["Target"].notnull() &
        panel_clean["Filter_Pass"]
    ].copy()

    print("【步驟一】執行無洩漏 Purged Cross-Validation 與機率校準...")
    tscv = TimeSeriesSplit(n_splits=5)
    hist_filtered["OOF_Prob"] = np.nan
    unique_hist_dates = hist_filtered.index.unique().sort_values()

    for tr_idx, va_idx in tscv.split(unique_hist_dates):
        tr_dates, va_dates = unique_hist_dates[tr_idx], unique_hist_dates[va_idx]
        tr_purged = tr_dates[tr_dates < (va_dates.min() - pd.Timedelta(days=15))]
        tr_mask = hist_filtered.index.isin(tr_purged if len(tr_purged) > 0 else tr_dates)
        va_mask = hist_filtered.index.isin(va_dates)

        X_tr, y_tr = hist_filtered.loc[tr_mask, f_cols], hist_filtered.loc[tr_mask, "Target"]
        if len(X_tr) > 0 and len(np.unique(y_tr)) > 1:
            cv_mod = lgb.LGBMClassifier(
                n_estimators=180, learning_rate=0.03, max_depth=5, num_leaves=24,
                subsample=0.8, colsample_bytree=0.8, class_weight="balanced", verbose=-1, random_state=42
            )
            cv_mod.fit(X_tr, y_tr)
            hist_filtered.loc[va_mask, "OOF_Prob"] = cv_mod.predict_proba(hist_filtered.loc[va_mask, f_cols])[:, 1]

    oof_valid = hist_filtered.dropna(subset=["OOF_Prob", "Target"])

    if len(oof_valid) >= 500:
        iso_calibrator = IsotonicRegression(out_of_bounds="clip")
        iso_calibrator.fit(oof_valid["OOF_Prob"], oof_valid["Target"])
        transform_prob = lambda x: iso_calibrator.transform(x)
    else:
        platt_calibrator = LogisticRegression(C=1.0, solver="lbfgs")
        platt_calibrator.fit(oof_valid[["OOF_Prob"]].values, oof_valid["Target"])
        transform_prob = lambda x: platt_calibrator.predict_proba(np.array(x).reshape(-1, 1))[:, 1]

    hist_high_conf = hist_filtered[hist_filtered["OOF_Prob"] >= CONFIDENCE_THRESHOLD].copy()
    if not hist_high_conf.empty:
        stock_stats_raw = hist_high_conf.groupby("Ticker").agg(
            count=("Target", "count"), wins=("Target", "sum"), mean_win=("Target", "mean")
        ).reset_index()
        stock_stats_raw["bayes_win_rate"] = (stock_stats_raw["wins"] + 2) / (stock_stats_raw["count"] + 5)
    else:
        stock_stats_raw = pd.DataFrame(columns=["Ticker", "count", "wins", "mean_win", "bayes_win_rate"])

    print("【步驟二】訓練全量最終模型與演算機構級風控指標...")
    full_model = lgb.LGBMClassifier(
        n_estimators=180, learning_rate=0.03, max_depth=5, num_leaves=24,
        subsample=0.8, colsample_bytree=0.8, class_weight="balanced", verbose=-1, random_state=42
    )
    full_model.fit(hist_filtered[f_cols], hist_filtered["Target"])

    recent_5_dates = unique_dates[-5:]
    recent_df = panel_clean[panel_clean.index.isin(recent_5_dates) & panel_clean["Filter_Pass"]].copy()

    if not recent_df.empty:
        recent_df["raw_prob"] = full_model.predict_proba(recent_df[f_cols])[:, 1]
        recent_df["calibrated_prob"] = transform_prob(recent_df["raw_prob"].values)

        top_targets = recent_df[recent_df["raw_prob"] >= CONFIDENCE_THRESHOLD].copy()

        if not top_targets.empty:
            top_targets["預測日期"] = top_targets.index.strftime("%Y-%m-%d")
            result_df = top_targets.merge(stock_stats_raw, on="Ticker", how="left")

            result_df["股票代號"] = result_df["Ticker"]
            result_df["股票名稱"] = result_df["Stock_Name"]
            result_df["收盤價"] = result_df["Close"].round(2)
            result_df["模型預測漲升機率"] = (result_df["raw_prob"] * 100).round(2).astype(str) + "%"
            result_df["外資買賣超比"] = (result_df["Foreign_Net_Vol_Ratio"] * 100).round(2).astype(str) + "%"
            result_df["外資連買天數"] = result_df["Foreign_Buy_Streak"].astype(int)

            def format_win_rate(row):
                cnt = row.get("count", 0)
                win = row.get("mean_win", 0)
                if pd.isna(cnt) or cnt == 0:
                    return "無歷史紀錄"
                elif cnt < 5:
                    return f"{win*100:.2f}% (樣本不足<5次)"
                else:
                    return f"{win*100:.2f}%"

            result_df["個股歷史勝率"] = result_df.apply(format_win_rate, axis=1)

            result_df["sample_weight"] = np.clip(result_df["count"].fillna(0) / 15.0, 0.1, 0.4)
            result_df["p_hist_num"] = result_df["bayes_win_rate"].fillna(0.38)
            result_df["p_calibrated"] = (result_df["calibrated_prob"] * (1.0 - result_df["sample_weight"])) + (result_df["p_hist_num"] * result_df["sample_weight"])

            dyn_win_pct = np.clip((result_df["ATR_14"] * 2.5) / result_df["Close"], 0.05, 0.18)
            dyn_atr_mult = np.where(result_df["Hist_Vol_20"] > 0.40, 1.5, np.where(result_df["Hist_Vol_20"] < 0.20, 1.0, 1.2))
            dyn_loss_pct = np.clip((result_df["ATR_14"] * dyn_atr_mult) / result_df["Close"], 0.03, 0.08)

            result_df["ev_num"] = (result_df["p_calibrated"] * (dyn_win_pct - TRANSACTION_COST_PCT)) - ((1.0 - result_df["p_calibrated"]) * (dyn_loss_pct + TRANSACTION_COST_PCT))
            result_df["單筆期望值(EV)"] = (result_df["ev_num"] * 100).round(2).astype(str) + "%"

            sharpe_est = (result_df["ev_num"] / (result_df["Hist_Vol_20"] + 1e-6)) * np.sqrt(252 / 10)
            result_df["夏普期望展望值"] = sharpe_est.round(2)

            b = (dyn_win_pct - TRANSACTION_COST_PCT) / (dyn_loss_pct + TRANSACTION_COST_PCT)
            p = result_df["p_calibrated"]
            q = 1.0 - p
            kelly_full = (b * p - q) / (b + 1e-6)

            mkt_risk_adj = np.where(result_df["Market_Vol_20"] > 0.25, 0.7, 1.0)
            result_df["kelly_num"] = np.maximum(0.0, (kelly_full / 2.0) * mkt_risk_adj)
            result_df["半凱利建議下注(%)"] = (result_df["kelly_num"] * 100).round(2).astype(str) + "%"

            result_df["產業分類"] = result_df.apply(lambda r: classify_industry(r["股票代號"], r["股票名稱"]), axis=1)

            MAX_IND_CAP = 30.0
            tot_k = result_df["kelly_num"].sum()
            result_df["raw_alloc_pct"] = (result_df["kelly_num"] / tot_k * 100.0) if tot_k > 0 else 0.0

            ind_sums = result_df.groupby("產業分類")["raw_alloc_pct"].transform("sum")
            capped_industries = result_df[ind_sums > MAX_IND_CAP]["產業分類"].unique()
            total_capped_allocated = MAX_IND_CAP * len(capped_industries)

            uncapped_mask = ~result_df["產業分類"].isin(capped_industries)
            remaining_target_pct = 100.0 - total_capped_allocated
            uncapped_current_sum = result_df.loc[uncapped_mask, "raw_alloc_pct"].sum()

            if uncapped_current_sum > 0 and remaining_target_pct > 0:
                rebalance_factor = remaining_target_pct / uncapped_current_sum
                result_df["final_alloc_num"] = result_df.apply(
                    lambda r: r["raw_alloc_pct"] * rebalance_factor if uncapped_mask.loc[r.name] else r["raw_alloc_pct"] * (MAX_IND_CAP / ind_sums.loc[r.name]), axis=1
                )
            else:
                result_df["final_alloc_num"] = result_df["raw_alloc_pct"]

            result_df["風控頂格再平衡建議部位比率"] = result_df["final_alloc_num"].round(2).astype(str) + "%"

            result_df["建議動態停利點(TP)"] = (result_df["Close"] + (result_df["ATR_14"] * 2.5)).round(2)
            result_df["建議動態停損點(SL)"] = (result_df["Close"] - (result_df["ATR_14"] * pd.Series(dyn_atr_mult, index=result_df.index))).round(2)

            result_df["距離停利幅(%)"] = (((result_df["建議動態停利點(TP)"] - result_df["Close"]) / result_df["Close"]) * 100).round(2).astype(str) + "%"
            result_df["距離停損幅(%)"] = (((result_df["建議動態停損點(SL)"] - result_df["Close"]) / result_df["Close"]) * 100).round(2).astype(str) + "%"

            result_df["波動風險評級"] = result_df["Hist_Vol_20"].apply(lambda hv: "低波動(穩健)" if hv < 0.20 else ("中波動(標準)" if hv < 0.35 else ("高波動(積極)" if hv < 0.50 else "極高波動(投機)")))

            def assign_signal_level(row):
                prob = row["raw_prob"]
                ev = row["ev_num"]
                if prob >= 0.70 and ev >= 0.03:
                    return "Strong Buy (強力推薦)"
                elif prob >= 0.58 and ev > 0:
                    return "Buy (偏多操作)"
                else:
                    return "Neutral (觀察待變)"

            result_df["綜合交易訊號等級"] = result_df.apply(assign_signal_level, axis=1)

            rr_ratio = (dyn_win_pct - TRANSACTION_COST_PCT) / (dyn_loss_pct + TRANSACTION_COST_PCT)
            result_df["風報比(Reward/Risk)"] = rr_ratio.round(2)

            buy_low = (result_df["Close"] - (result_df["ATR_14"] * 0.4)).round(2)
            buy_high = result_df["Close"].round(2)
            result_df["建議逢低進場買進區間"] = buy_low.astype(str) + " ~ " + buy_high.astype(str)

            alloc_ratio = result_df["final_alloc_num"] / 100.0
            allocated_cash = PORTFOLIO_BASE_CAPITAL * alloc_ratio
            result_df["預估單筆最大虧損金額(萬)"] = ((allocated_cash * (dyn_loss_pct + TRANSACTION_COST_PCT)) / 10000.0).round(2)
            result_df["預估單筆期望獲利金額(萬)"] = ((allocated_cash * result_df["ev_num"]) / 10000.0).round(2)

            result_df["大盤總體風險狀態"] = np.where(result_df["Market_Vol_20"] > 0.25, "高風險(減半下注)", "低風險(正常配置)")
            result_df["外資5日籌碼集中度"] = (result_df["Foreign_Net_MA5"] * 100).round(2).astype(str) + "%"

            avg_vol_shares = result_df["Vol_MA5"] * 0.02
            avg_vol_shares = avg_vol_shares.replace([np.inf, -np.inf], np.nan).fillna(1)
            result_df["單日建議最大交易張數上限"] = np.maximum(1, avg_vol_shares.round(0)).astype(int)

            total_vol_weighted = (result_df["final_alloc_num"] * result_df["Hist_Vol_20"]).sum()
            result_df["個股風險貢獻度(%)"] = (((result_df["final_alloc_num"] * result_df["Hist_Vol_20"]) / (total_vol_weighted + 1e-6)) * 100).round(2).astype(str) + "%"

            var_95 = allocated_cash * 1.645 * result_df["Hist_Vol_20"] * np.sqrt(10 / 252)
            result_df["VAR_95_萬"] = (var_95 / 10000.0).round(2)

            sortino_est = (result_df["ev_num"] / (dyn_loss_pct + 1e-6)) * np.sqrt(252 / 10)
            result_df["卡爾瑪比率(Sortino/Calmar Est.)"] = sortino_est.round(2)

            result_df["建議分批進場次數"] = result_df["Hist_Vol_20"].apply(lambda hv: "分4-5批建倉 (防止劇烈滑點)" if hv > 0.45 else ("分2-3批建倉 (標準分批)" if hv > 0.28 else "單筆或分2批進場"))
            result_df["主力/外資籌碼共振訊號"] = result_df.apply(lambda r: "雙雄強勢買超" if r["Foreign_Buy_Streak"] >= 3 and r["Foreign_Net_MA5"] > 0.05 else ("法人持續布局" if r["Foreign_Buy_Streak"] >= 1 else "籌碼觀察中"), axis=1)
            result_df["建議停損點相對ATR倍數"] = pd.Series(dyn_atr_mult, index=result_df.index).round(2).astype(str) + "x ATR"

            result_df["流動性風險評級"] = np.where(result_df["Turnover_MA5"] > 500_000_000, "高流動性(大額無礙)", np.where(result_df["Turnover_MA5"] > 100_000_000, "中流動性(標準)", "低流動性(需留意滑點)"))
            result_df["法人/大戶鎖碼強弱度"] = np.where(result_df["Foreign_Buy_Streak"] >= 3, "強力鎖碼(★★★★★)", "穩定佈局(★★★☆☆)")

            # 技術面基礎策略判定
            result_df["建议策略_raw"] = np.where(
                result_df["Close"] > result_df["MA60"],
                "右側突破買進 / 順勢加碼",
                "左側逢低分批打底 / 黃金右腳佈局"
            )

            result_df["多空情緒分水嶺乖離 (%)"] = (result_df["BIAS_5"] * 100).round(2).astype(str) + "%"
            liq_factor = np.clip(result_df["Turnover_MA5"] / 100_000_000, 0.1, 2.0)
            result_df["流動性風控係數"] = liq_factor.round(2)

            # ==============================================================================
            # 💡 嵌入機構級風控邏輯關斷與 5 個新增欄位 (Institutional Post-Processing)
            # ==============================================================================

            # 1. 自動區分交易計價幣別 (TWD vs USD)
            result_df["交易幣別"] = result_df["股票代號"].apply(
                lambda x: "TWD" if (".TW" in str(x).upper() or ".TWO" in str(x).upper() or str(x).isdigit()) else "USD"
            )

            # 2. 硬性風控關斷 (Risk Gate Intervention)
            # 當 EV <= 0 或 凱利比例 == 0 或 漲升機率 < 50% 時執行阻斷
            blocked_mask = (result_df["ev_num"] <= 0) | (result_df["kelly_num"] <= 0) | (result_df["raw_prob"] < 0.50)

            result_df["風控執行狀態"] = "PASS (允許交易)"
            result_df.loc[blocked_mask, "風控執行狀態"] = "BLOCKED (風控阻斷)"

            # 策略強制關斷重置
            result_df["建議實戰進場策略"] = result_df["建议策略_raw"]
            result_df.loc[blocked_mask, "建議實戰進場策略"] = "觀望待變 / 禁忌建倉"
            result_df.loc[blocked_mask, "半凱利建議下注(%)"] = "0.0%"
            result_df.loc[blocked_mask, "風控頂格再平衡建議部位比率"] = "0.0%"

            # 3. 美股籌碼例外動態調整
            us_mask = result_df["交易幣別"] == "USD"
            result_df.loc[us_mask, "主力/外資籌碼共振訊號"] = "不適用 (美股數據未串接)"
            result_df.loc[us_mask, "法人/大戶鎖碼強弱度"] = "不適用 (美股數據未串接)"

            # 4. 新增欄位計算：動態停損%
            result_df["動態停損%"] = (((result_df["Close"] - result_df["建議動態停損點(SL)"]) / result_df["Close"]) * -100).round(2).astype(str) + "%"

            # 5. 新增欄位計算：建議持倉週期
            def assign_holding_period(vol_rating):
                if "極高" in vol_rating or "高" in vol_rating:
                    return "極短線 (1-3日衝刺)"
                elif "中" in vol_rating:
                    return "波段 (2-4週順勢)"
                else:
                    return "中長線 (1-3個月趨勢)"
            result_df["建議持倉週期"] = result_df["波動風險評級"].apply(assign_holding_period)

            # 6. 新增欄位計算：建議絕對交易下單量 (單位自動轉換張/股)
            def calc_exact_order_size(row):
                if row["風控執行狀態"] == "BLOCKED (風控阻斷)":
                    return "0"

                kelly_pct = float(str(row["半凱利建議下注(%)"]).rstrip("%")) / 100.0
                allocated_fund_twd = PORTFOLIO_BASE_CAPITAL * kelly_pct

                if row["交易幣別"] == "TWD":
                    shares = int(allocated_fund_twd // (row["Close"] * 1000.0))
                    return f"{shares:,} 張"
                else:
                    usd_fund = allocated_fund_twd / 32.0  # 預設 USD/TWD 匯率 32
                    shares = int(usd_fund // row["Close"])
                    return f"{shares:,} 股"

            result_df["建議絕對交易下單量"] = result_df.apply(calc_exact_order_size, axis=1)

            # ==============================================================================
            # 7. 最終產出欄位編排 (原 37 個欄位 + 5 個新增擴充欄位 = 共 42 個欄位)
            # ==============================================================================
            final_cols = [
                # 原有 37 個欄位
                "預測日期", "股票代號", "股票名稱", "產業分類", "收盤價", "建議逢低進場買進區間",
                "綜合交易訊號等級", "模型預測漲升機率", "個股歷史勝率", "外資買賣超比",
                "外資5日籌碼集中度", "外資連買天數", "單筆期望值(EV)", "夏普期望展望值",
                "風報比(Reward/Risk)", "波動風險評級", "建議動態停利點(TP)", "距離停利幅(%)",
                "建議動態停損點(SL)", "距離停損幅(%)", "半凱利建議下注(%)",
                "風控頂格再平衡建議部位比率", "預估單筆最大虧損金額(萬)", "預估單筆期望獲利金額(萬)",
                "單日建議最大交易張數上限", "大盤總體風險狀態", "個股風險貢獻度(%)", "VAR_95_萬",
                "卡爾瑪比率(Sortino/Calmar Est.)", "建議分批進場次數", "主力/外資籌碼共振訊號",
                "建議停損點相對ATR倍數", "流動性風險評級", "法人/大戶鎖碼強弱度", "建議實戰進場策略",
                "多空情緒分水嶺乖離 (%)", "流動性風控係數",
                # 新增發揮之 5 個專業欄位
                "交易幣別", "風控執行狀態", "動態停損%", "建議持倉週期", "建議絕對交易下單量"
            ]

            output_df = result_df[final_cols].sort_values(by=["預測日期", "模型預測漲升機率"], ascending=[False, False])

            print(f"\n[近五日預測與機構級風控系統結果 (全功能優化版)] (門檻 >= {CONFIDENCE_THRESHOLD*100:.0f}%)\n")
            display(output_df)

            fname = "recent_5days_institutional_signals_hybrid_pro.xlsx"
            output_df.to_excel(fname, index=False)
            print(f"\n💾 混血策略評估結果已匯出至 Excel 檔案：{fname}")
            if HAS_COLAB:
                files.download(fname)
        else:
            print("近五日無符合信心門檻條件的標的。")
    else:
        print("近五日無符合條件標的。")

    gc.collect()


【系統初始化】下載雙市場大盤數據與籌碼庫...


【資料建構】載入與計算全套因子面板資料 (支援左右側混血策略)...
【步驟一】執行無洩漏 Purged Cross-Validation 與機率校準...
【步驟二】訓練全量最終模型與演算機構級風控指標...

[近五日預測與機構級風控系統結果 (全功能優化版)] (門檻 >= 58%)



,預測日期,股票代號,股票名稱,產業分類,收盤價,建議逢低進場買進區間,綜合交易訊號等級,模型預測漲升機率,個股歷史勝率,外資買賣超比,...,流動性風險評級,法人/大戶鎖碼強弱度,建議實戰進場策略,多空情緒分水嶺乖離 (%),流動性風控係數,交易幣別,風控執行狀態,動態停損%,建議持倉週期,建議絕對交易下單量
46,2026-09-11,GEN,Gen Digital,美股科技/軟體資安,30.26,29.92 ~ 30.26,Neutral (觀察待變),63.64%,0.00%,0.0%,...,中流動性(標準),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,0.56%,1.44,USD,BLOCKED (風控阻斷),-3.4%,波段 (2-4週順勢),0
47,2026-09-11,OKTA,Okta,美股科技/半導體,166.50,162.47 ~ 166.5,Buy (偏多操作),59.06%,57.89%,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),右側突破買進 / 順勢加碼,-1.89%,2.00,USD,PASS (允許交易),-9.09%,極短線 (1-3日衝刺),4 股
45,2026-09-11,8050.TWO,廣積,台股電子/半導體/供應鏈,56.10,55.56 ~ 56.1,Neutral (觀察待變),58.47%,28.00%,0.0%,...,低流動性(需留意滑點),穩定佈局(★★★☆☆),觀望待變 / 禁忌建倉,0.57%,0.40,TWD,BLOCKED (風控阻斷),-2.91%,波段 (2-4週順勢),0
48,2026-09-11,MSFT,Microsoft 微軟,美股科技/半導體,495.63,491.61 ~ 495.63,Neutral (觀察待變),58.1%,無歷史紀錄,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,0.19%,2.00,USD,BLOCKED (風控阻斷),-2.44%,波段 (2-4週順勢),0
44,2026-09-10,OKTA,Okta,美股科技/半導體,171.11,167.18 ~ 171.11,Buy (偏多操作),73.71%,57.89%,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),右側突破買進 / 順勢加碼,0.36%,2.00,USD,PASS (允許交易),-8.63%,極短線 (1-3日衝刺),2 股
37,2026-09-10,AAPL,Apple 蘋果,美股科技/半導體,326.57,323.56 ~ 326.57,Neutral (觀察待變),68.32%,8.33%,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,1.65%,2.00,USD,BLOCKED (風控阻斷),-2.76%,波段 (2-4週順勢),0
42,2026-09-10,GEN,Gen Digital,美股科技/軟體資安,29.97,29.61 ~ 29.97,Neutral (觀察待變),64.43%,0.00%,0.0%,...,中流動性(標準),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,-1.1%,1.55,USD,BLOCKED (風控阻斷),-3.57%,極短線 (1-3日衝刺),0
38,2026-09-10,MSFT,Microsoft 微軟,美股科技/半導體,492.44,488.38 ~ 492.44,Neutral (觀察待變),62.76%,無歷史紀錄,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,-1.03%,2.00,USD,BLOCKED (風控阻斷),-2.47%,波段 (2-4週順勢),0
43,2026-09-10,FTNT,Fortinet,美股科技/半導體,158.85,155.98 ~ 158.85,Neutral (觀察待變),61.05%,33.33%,0.0%,...,中流動性(標準),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,1.03%,2.00,USD,BLOCKED (風控阻斷),-6.77%,極短線 (1-3日衝刺),0
39,2026-09-10,JNJ,Johnson & Johnson 嬌生,美股醫療保健/生技製藥,266.35,264.24 ~ 266.35,Neutral (觀察待變),60.26%,10.00%,0.0%,...,高流動性(大額無礙),不適用 (美股數據未串接),觀望待變 / 禁忌建倉,-1.8%,2.00,USD,BLOCKED (風控阻斷),-2.38%,波段 (2-4週順勢),0



💾 混血策略評估結果已匯出至 Excel 檔案：recent_5days_institutional_signals_hybrid_pro.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import gc
import logging
import os
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import yfinance as yf
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

try:
    from IPython.display import display
except ImportError:
    def display(df):
        print(df.to_string())

CACHE_DIR = "./chip_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.58  # 門檻：捕捉發動訊號
PORTFOLIO_BASE_CAPITAL_TWD = 10_000_000  # 預設台股總本金：NTD 1,000 萬
PORTFOLIO_BASE_CAPITAL_USD = 300_000     # 預設美股總本金：USD 30 萬
TRANSACTION_COST_PCT = 0.004

http_session = requests.Session()
retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
http_session.mount("https://", HTTPAdapter(max_retries=retries))

# ==============================================================================
# 0. 股票資料池定義 (完整保持 160 隻標的)
# ==============================================================================
stock_dict = {
    "0050.TW": "元大台灣50", "0056.TW": "元大高股息", "00878.TW": "國泰永續高股息", "00770.TW": "國泰北美科技", "00981A.TW": "統一台股增長主動式", "SPCX": "SPACs ETF", "SOXX": "iShares半導體ETF", "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果", "GOOG": "Google / Alphabet", "META": "Meta", "MSFT": "Microsoft 微軟", "NVDA": "NVIDIA 輝達", "TSM": "台積電 ADR", "TSLA": "Tesla 特斯拉", "ENTG": "Entegris 英特格", "SMR": "NuScale Power 小型核反應爐", "BE": "Bloom Energy 燃料電池", "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾", "AMAT": "Applied Materials 應用材料", "LRCX": "Lam Research 柯林研發", "KLAC": "KLA 科磊", "AMD": "AMD 超微", "AVGO": "Broadcom 博通", "QCOM": "Qualcomm 高通", "INTC": "Intel 英特爾", "MU": "Micron 鎂光", "TXN": "Texas Instruments 德州儀器", "ARM": "ARM 晶心/安謀", "MRVL": "Marvell 邁威爾", "ADI": "Analog Devices 亞德諾", "MPWR": "Monolithic Power 芯源系統", "ON": "ON Semiconductor 安森美", "SWKS": "Skyworks 思佳訊", "QRVO": "Qorvo 威訊", "TER": "Teradyne 泰瑞達", "MKSI": "MKS Instruments",
    "PANW": "Palo Alto Networks", "CRWD": "CrowdStrike", "FTNT": "Fortinet", "NET": "Cloudflare", "ZS": "Zscaler", "OKTA": "Okta", "S": "SentinelOne", "GEN": "Gen Digital", "RPD": "Rapid7", "CBRS": "CyberArk",
    "2471.TW": "資通", "2480.TW": "敦陽科", "3029.TW": "零壹", "6214.TW": "精誠", "3130.TW": "一零四", "2427.TW": "三商電", "3027.TW": "盛達", "5203.TW": "訊連", "5471.TW": "松翰", "5410.TW": "國統", "6183.TW": "關貿", "6203.TWO": "海韻電", "6210.TWO": "慶生", "6593.TWO": "台灣銘板", "6689.TW": "伊雲谷", "6690.TWO": "安碁資訊", "6752.TWO": "睿嘉", "6763.TWO": "綠界科技", "6865.TWO": "偉康科技", "6874.TWO": "倍力", "6928.TW": "全達",
    "2382.TW": "廣達", "3231.TW": "緯創", "6669.TW": "緯穎", "2317.TW": "鴻海", "2356.TW": "英業達", "2324.TW": "仁寶", "2376.TW": "技嘉", "3706.TW": "神達", "2377.TW": "微 MSI", "2357.TW": "華碩", "4938.TW": "和碩", "3005.TW": "神基", "2353.TW": "宏碁",
    "2330.TW": "台積電", "2303.TW": "聯電", "2454.TW": "聯發科", "3034.TW": "聯詠", "3661.TW": "世芯-KY", "3443.TW": "創意", "4961.TW": "天鈺", "6415.TW": "矽力-KY", "6531.TW": "愛普*", "3035.TW": "智原", "6643.TWO": "M31", "4966.TWO": "譜瑞-KY", "5269.TW": "祥碩", "6104.TWO": "創唯", "6756.TW": "威鋒電子",
    "2342.TW": "茂矽", "6770.TW": "力積電", "3707.TWO": "漢磊", "3016.TW": "嘉晶", "3711.TW": "日月光投控", "2449.TW": "京元電子", "6257.TW": "矽格", "3264.TWO": "欣銓", "6239.TW": "力成", "2329.TW": "華泰", "2441.TW": "超豐",
    "3131.TWO": "弘塑", "3583.TW": "辛耘", "6187.TWO": "萬潤", "2467.TW": "志聖", "8027.TWO": "钛昇", "5434.TW": "崇越", "3010.TW": "華立", "1560.TW": "中砂", "3680.TWO": "家登", "5234.TW": "達興材料", "4749.TWO": "新應材", "8028.TW": "昇陽半導體", "6515.TW": "穎崴", "6683.TWO": "雍智科技", "6510.TWO": "精測", "6223.TWO": "旺矽",
    "2404.TW": "漢唐", "1773.TW": "勝一", "6196.TW": "帆宣", "6139.TW": "亞翔", "6613.TWO": "朋億*", "4755.TW": "三福化", "4768.TW": "晶呈科技", "3563.TW": "牧德", "3167.TW": "大量", "6438.TW": "迅得", "1595.TWO": "川寶",
    "6147.TWO": "頎邦", "8150.TW": "南茂", "6552.TW": "易華電", "5536.TWO": "聖暉*", "3644.TWO": "凌嘉科", "7769.TW": "鴻勁",
    "2344.TW": "華邦電", "2408.TW": "南亞科", "2337.TW": "旺宏", "3006.TW": "晶豪科", "3260.TWO": "威剛", "2451.TW": "創見", "4967.TW": "十銓", "8271.TW": "宇瞻", "5289.TWO": "宜晶", "8299.TWO": "群聯", "5351.TWO": "鈺創",
    "2308.TW": "台達電", "2301.TW": "光寶科", "6282.TW": "康舒", "6412.TW": "群電", "3665.TW": "貿聯-KY", "3017.TW": "奇鋐", "3324.TWO": "雙鴻", "3653.TW": "健策", "2421.TW": "建準", "8996.TW": "高力", "3483.TWO": "力致", "6230.TW": "尼得科超眾", "3013.TW": "晟銘電", "6805.TW": "富世達", "8210.TW": "勤誠", "6117.TW": "迎廣", "6235.TW": "華孚", "2354.TW": "鴻準", "3376.TW": "新日興", "3548.TWO": "兆利", "5243.TW": "乙盛-KY", "6715.TW": "嘉基", "3533.TW": "嘉澤", "3217.TWO": "優群", "3023.TW": "信邦", "2392.TW": "正崴", "3689.TWO": "湧德", "3357.TWO": "臺慶科", "6862.TW": "三集瑞-KY", "6821.TWO": "聯寶", "3207.TWO": "耀勝", "6197.TW": "佳必股", "8103.TW": "瀚荃", "3526.TWO": "凡甲", "3605.TW": "宏致", "2059.TW": "川湖", "6584.TWO": "南俊國際",
    "2327.TW": "國巨", "2492.TW": "華新科", "2375.TW": "凱美", "2478.TW": "大毅", "3026.TW": "禾伸堂", "3090.TW": "日電貿", "6173.TWO": "信昌電", "6155.TW": "鈞寶", "6175.TWO": "立敦", "5328.TWO": "華容", "3236.TWO": "千如", "8043.TWO": "蜜望實",
    "3037.TW": "欣興", "8046.TW": "南電", "3189.TW": "景碩", "4958.TW": "臻鼎-KY", "2368.TW": "金像電", "3044.TW": "健鼎", "2313.TW": "華通", "8155.TWO": "博智", "2383.TW": "台光電", "6274.TWO": "台燿", "6213.TW": "聯茂", "1717.TW": "長興", "1815.TWO": "富喬", "1802.TW": "台玻", "5340.TWO": "建榮", "5475.TWO": "德宏", "3305.TW": "昇貿", "3631.TWO": "晟楠", "8358.TWO": "金居", "8021.TW": "尖點", "6672.TW": "騰輝電子-KY",
    "2345.TW": "智邦", "5388.TW": "中磊", "3558.TWO": "神準", "3704.TW": "合勤控", "4906.TW": "正文", "4979.TWO": "華星光", "6442.TW": "光聖", "4908.TWO": "前鼎", "3163.TWO": "波若威", "3450.TW": "聯鈞", "6426.TW": "統新", "4977.TW": "眾達-KY", "6530.TWO": "創威", "3363.TWO": "上詮", "3234.TWO": "光環", "4903.TWO": "聯光通", "3081.TWO": "聯亞", "4991.TWO": "環宇-KY", "4971.TWO": "IET-KY", "6588.TWO": "東典光電", "3491.TWO": "昇達科", "2314.TW": "台揚", "6285.TW": "啟碁", "3105.TWO": "穩懋", "2455.TW": "全新", "3138.TW": "耀登", "2419.TW": "仲琦",
    "2395.TW": "研華", "6166.TW": "凌華", "8050.TWO": "廣積", "3556.TWO": "禾瑞亞", "2414.TW": "精技", "6414.TW": "樺漢", "3022.TW": "威強電", "2397.TW": "友通", "5314.TWO": "世紀",
    "6781.TW": "AES-KY", "3211.TWO": "順達", "6121.TWO": "新普", "3323.TWO": "加百裕", "3625.TWO": "西勝", "8038.TWO": "長園科", "4931.TWO": "新盛力",
    "1519.TW": "華城", "1513.TW": "中興電", "1514.TW": "亞力", "1503.TW": "士電", "1609.TW": "大亞", "1605.TW": "華新", "1608.TW": "華榮", "6869.TW": "雲豹能源", "2049.TW": "上銀", "4576.TW": "大銀微系統", "4585.TW": "達明", "2359.TW": "所羅門", "6188.TWO": "廣明", "8374.TW": "羅昇", "5443.TWO": "均豪", "6640.TWO": "均華", "2464.TW": "盟立", "6215.TW": "和椿", "4562.TW": "穎漢", "1590.TW": "亞德客-KY", "1504.TW": "東元",
    "3481.TW": "群創", "2409.TW": "友達", "3008.TW": "大立光", "4915.TW": "先進光", "5288.TW": "匯鑽科", "2393.TW": "億光",
    "2201.TW": "裕隆", "2204.TW": "中華", "2206.TW": "三陽工業", "1536.TW": "和大", "2231.TW": "聯嘉", "3552.TWO": "同致", "6279.TWO": "胡連",
    "2603.TW": "長榮", "2609.TW": "陽明", "2615.TW": "萬海", "2605.TW": "新興", "2606.TW": "裕民", "2612.TW": "中航", "2617.TW": "台航", "2637.TW": "慧洋-KY", "2641.TWO": "正德", "5608.TW": "四維航", "2610.TW": "華航", "2618.TW": "長榮航", "2630.TW": "亞航", "5603.TWO": "陸海", "2607.TW": "勞運", "2608.TW": "嘉里大榮", "2611.TW": "志信", "2613.TW": "中櫃", "2636.TW": "台驊投控", "2642.TW": "宅配通", "2633.TW": "台灣高鐵", "5607.TW": "遠雄港", "5609.TWO": "中菲行", "8367.TW": "建新國際",
    "2892.TW": "第一金", "5880.TW": "合庫金", "1210.TW": "大成", "1215.TW": "卜蜂", "1216.TW": "統一", "2912.TW": "統一超", "5903.TWO": "全家", "1303.TW": "南亞", "2465.TW": "麗臺", "8163.TW": "達方", "3042.TW": "晶技", "8182.TWO": "加高", "3229.TW": "泰藝", "3308.TW": "聯傑", "6284.TWO": "佳邦", "2484.TW": "希華", "8088.TWO": "華信科",
}

download_cache = {}

def classify_industry(code, name):
    code_str = str(code).upper()
    name_str = str(name)
    if code_str in ["JNJ"]:
        return "美股醫療保健/生技製藥"
    elif code_str in ["GEN"]:
        return "美股科技/軟體資安"
    elif code_str.startswith("00") or "ETF" in name_str or "增長" in name_str or code_str in ["SPCX", "SOXX", "SMH"]:
        return "ETF/大盤指數"
    elif (code_str.startswith("28") or code_str.startswith("58")) and ("金" in name_str or "銀" in name_str or "保" in name_str):
        return "金融/保險"
    elif code_str.startswith("26") or code_str.startswith("56") or "航" in name_str or "海" in name_str or "宅配" in name_str or "高鐵" in name_str or "大榮" in name_str or "港" in name_str:
        return "航運/物流"
    elif code_str.startswith("12") or code_str.startswith("29") or code_str.startswith("59") or "統一超" in name_str or "全家" in name_str or "食品" in name_str:
        return "食品/民生消費"
    elif code_str in ["AAPL", "META", "GOOG", "MSFT", "NVDA", "TSM", "ASML", "AMAT", "LRCX", "KLAC", "AMD", "AVGO", "QCOM", "INTC", "MU", "TXN", "ARM", "MRVL", "ADI", "MPWR", "ON", "SWKS", "QRVO", "TER", "MKSI", "PANW", "CRWD", "FTNT", "NET", "ZS", "OKTA", "S", "RPD", "CBRS", "ENTG", "SMR", "BE"]:
        return "美股科技/半導體"
    else:
        return "台股電子/半導體/供應鏈"

def fetch_twse_foreign_bulk(date_str):
    cache_file = os.path.join(CACHE_DIR, f"twse_{date_str}.json")
    if os.path.exists(cache_file):
        try: return pd.read_json(cache_file)
        except Exception: pass
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={date_str}&selectType=ALLBUT0999&response=json"
    try:
        time.sleep(0.15)
        res = http_session.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5, verify=False)
        if res.status_code == 200:
            js = res.json()
            if js.get("stat") == "OK":
                df = pd.DataFrame(js["data"], columns=js["fields"])
                df.to_json(cache_file)
                return df
    except Exception: pass
    return None

def fetch_tpex_foreign_bulk(date_str_slash):
    date_clean = date_str_slash.replace("/", "")
    cache_file = os.path.join(CACHE_DIR, f"tpex_{date_clean}.json")
    if os.path.exists(cache_file):
        try: return pd.read_json(cache_file)
        except Exception: pass
    url = f"https://www.tpex.org.tw/www/zh-tw/insti/qfiiStat?type=Daily&date={date_str_slash}&searchType=buy&id=&response=json"
    try:
        time.sleep(0.15)
        res = http_session.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5, verify=False)
        if res.status_code == 200:
            js = res.json()
            if "tables" in js and len(js["tables"]) > 0:
                t = js["tables"][0]
                df = pd.DataFrame(t["data"], columns=t["fields"])
                df.to_json(cache_file)
                return df
    except Exception: pass
    return None

def sanitize_df(df):
    if df is None or df.empty: return None
    d = df.copy()
    if d.index.tz is not None: d.index = d.index.tz_localize(None)
    if isinstance(d.columns, pd.MultiIndex):
        for level in range(d.columns.nlevels):
            col_names = [str(c).strip().title() for c in d.columns.get_level_values(level)]
            if "Close" in col_names:
                d.columns = d.columns.get_level_values(level)
                break
        else: d.columns = d.columns.get_level_values(-1)
    col_map = {c: str(c).strip().title().replace("Adj Close", "Close") for c in d.columns}
    d = d.rename(columns=col_map)
    needed = ["Open", "Close", "High", "Low", "Volume"]
    return d[needed].dropna(subset=["Close"]) if all(k in d.columns for k in needed) else None

def compute_market_features(market_df):
    m = sanitize_df(market_df)
    if m is None: return pd.DataFrame()
    m_returns = m["Close"].pct_change()
    return pd.DataFrame({
        "Market_Vol_20": m_returns.rolling(20).std() * np.sqrt(252),
        "Market_Ret_20": m["Close"].pct_change(20),
        "Market_MA_Dist": (m["Close"] - m["Close"].rolling(20).mean()) / (m["Close"].rolling(20).mean() + 1e-6),
        "Market_Close": m["Close"],
    }, index=m.index)

def get_market_data():
    try: tw_feats = compute_market_features(yf.download("^TWII", period="3y", progress=False, auto_adjust=True))
    except Exception: tw_feats = compute_market_features(yf.download("0050.TW", period="3y", progress=False, auto_adjust=True))
    try: us_feats = compute_market_features(yf.download("^GSPC", period="3y", progress=False, auto_adjust=True))
    except Exception: us_feats = compute_market_features(yf.download("SPY", period="3y", progress=False, auto_adjust=True))
    return tw_feats, us_feats

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def download_stock_with_auto_suffix(ticker, period="1y"):
    if ticker in download_cache: return download_cache[ticker]
    pure_t = ticker.split(".")[0]
    candidates = [f"{pure_t}.TWO", f"{pure_t}.TW", ticker] if pure_t.isdigit() and len(pure_t) >= 4 else [ticker]
    candidates = list(dict.fromkeys(candidates))
    for cand in candidates:
        try:
            df = yf.download(cand, period=period, progress=False, auto_adjust=True)
            cleaned_df = sanitize_df(df)
            if cleaned_df is not None and len(cleaned_df) > 50:
                download_cache[ticker] = (cleaned_df, cand)
                return cleaned_df, cand
        except Exception: continue
    return None, ticker

def compute_features(df, market_feats, is_tw_stock=False, ticker=None, twse_cache=None, tpex_cache=None):
    d = df.copy()
    if d is None or len(d) < 200: return None, []
    if twse_cache is None: twse_cache = {}
    if tpex_cache is None: tpex_cache = {}

    x = np.arange(5)
    x_dev = x - x.mean()
    x_var = (x_dev**2).sum()
    close_vals = d["Close"].values
    if len(close_vals) >= 5:
        shape = (len(close_vals) - 5 + 1, 5)
        strides = (close_vals.strides[0], close_vals.strides[0])
        windows = np.lib.stride_tricks.as_strided(close_vals, shape=shape, strides=strides)
        y_dev = windows - windows.mean(axis=1, keepdims=True)
        slopes = (y_dev * x_dev).sum(axis=1) / x_var
        pad_slopes = np.concatenate([np.repeat(np.nan, 4), slopes])
        d["Close_Slope"] = pad_slopes / (d["Close"].values + 1e-6)
    else: d["Close_Slope"] = 0.0

    tr = pd.concat([
        d["High"] - d["Low"],
        (d["High"] - d["Close"].shift(1)).abs(),
        (d["Low"] - d["Close"].shift(1)).abs()
    ], axis=1).max(axis=1)
    d["ATR_14"] = tr.rolling(14).mean()
    d["NATR"] = d["ATR_14"] / (d["Close"] + 1e-6)
    d["Hist_Vol_20"] = d["Close"].pct_change().rolling(20).std() * np.sqrt(252)

    ma5, ma20, ma60 = d["Close"].rolling(5).mean(), d["Close"].rolling(20).mean(), d["Close"].rolling(60).mean()
    d["MA60"] = ma60
    d["MA_Bullish_Align"] = ((d["Close"] > ma5) & (ma5 > ma20) & (ma20 > ma60)).astype(float)
    std20 = d["Close"].rolling(20).std()
    d["BB_Bandwidth"] = (4 * std20) / (ma20 + 1e-6)
    d["BB_Squeeze"] = d["BB_Bandwidth"] / (d["BB_Bandwidth"].rolling(60).mean() + 1e-6)
    d["BIAS_5"] = (d["Close"] - ma5) / (ma5 + 1e-6)

    ema12, ema26 = d["Close"].ewm(span=12).mean(), d["Close"].ewm(span=26).mean()
    d["MACD_Hist"] = (ema12 - ema26 - (ema12 - ema26).ewm(span=9).mean()) / (d["Close"] + 1e-6)
    d["MACD_Hist_Slope"] = d["MACD_Hist"].diff(3)
    d["RSI_14"] = compute_rsi(d["Close"], 14) / 100.0
    d["RSI_Slope"] = d["RSI_14"].diff(3)

    mfv = (((d["Close"] - d["Low"]) - (d["High"] - d["Close"])) / (d["High"] - d["Low"] + 1e-6)) * d["Volume"]
    d["CMF_20"] = (mfv.rolling(20).sum() / (d["Volume"].rolling(20).sum() + 1e-6)).fillna(0)

    d["Volume_Explosion"] = np.clip(d["Volume"] / (d["Volume"].rolling(5).mean() + 1e-6), 0, 10)
    d["Turnover_Rate"] = np.clip(d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6), 0, 10)
    d["Body_Ratio"] = (d["Close"] - d["Open"]).abs() / (d["High"] - d["Low"] + 1e-6)

    d["Vol_MA5"] = d["Volume"].rolling(5).mean()
    d["Turnover_MA5"] = (d["Volume"] * d["Close"]).rolling(5).mean()

    d["Foreign_Net_Vol_Ratio"] = 0.0
    if is_tw_stock and ticker:
        pure_ticker = ticker.split(".")[0]
        f_dict = {}
        cache_source = twse_cache if ticker.endswith(".TW") else tpex_cache
        for dt, raw_dict in cache_source.items():
            if pure_ticker in raw_dict: f_dict[dt] = raw_dict[pure_ticker]
        if f_dict:
            f_series = pd.Series(f_dict)
            d["Foreign_Net_Vol_Ratio"] = f_series.reindex(d.index).fillna(0.0) / (d["Volume"] + 1e-6)

    d["Foreign_Net_MA5"] = d["Foreign_Net_Vol_Ratio"].rolling(5).mean().fillna(0)
    is_buy = (d["Foreign_Net_Vol_Ratio"] > 0).astype(int)
    is_not_buy = (d["Foreign_Net_Vol_Ratio"] <= 0).astype(int)
    d["Foreign_Buy_Streak"] = is_buy.groupby(is_not_buy.cumsum()).cumsum()

    d["Alpha_5d"] = d["Close"].pct_change(5) - market_feats["Market_Close"].pct_change(5)
    m_re = market_feats.reindex(d.index).ffill()
    ret_stock, ret_mkt = d["Close"].pct_change(), m_re["Market_Close"].pct_change()
    cov_sm = ret_stock.rolling(20).cov(ret_mkt)
    var_m = ret_mkt.rolling(20).var()
    d["Rolling_Beta"] = (cov_sm / (var_m + 1e-6)).fillna(1.0)
    d["Market_Vol_20"], d["Market_Ret_20"], d["Market_MA_Dist"] = m_re["Market_Vol_20"], m_re["Market_Ret_20"], m_re["Market_MA_Dist"]

    liquidity_ok = d["Turnover_MA5"] >= (30_000_000 if is_tw_stock else 2_000_000)
    right_side_trend = (d["Close"] > ma60) & ((ma20 - ma20.rolling(5).mean()).abs() / ma20 <= 0.04)
    left_side_reversal = (
        (d["Close"] <= ma60 * 1.08) &
        (
            ((d["RSI_14"] < 0.45) & (d["RSI_Slope"] > 0)) |
            (d["MACD_Hist_Slope"] > 0) |
            ((d["Volume_Explosion"] > 1.2) & (d["Body_Ratio"] > 0.4) & (d["Close"] > d["Open"]))
        )
    )

    d["Filter_Pass"] = liquidity_ok & (right_side_trend | left_side_reversal)

    upper_barrier = d["Close"] + (d["ATR_14"] * 2.5)
    lower_barrier = d["Close"] - (d["ATR_14"] * 1.5)
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=10)
    f_max = d["High"].shift(-1).rolling(indexer).max()
    f_min = d["Low"].shift(-1).rolling(indexer).min()
    d["Target"] = ((f_max >= upper_barrier) & (f_min > lower_barrier)).astype(float)
    d.iloc[-10:, d.columns.get_loc("Target")] = np.nan

    f_cols = [
        "Close_Slope", "NATR", "Hist_Vol_20", "BB_Bandwidth", "BB_Squeeze", "BIAS_5",
        "MA_Bullish_Align", "Volume_Explosion", "Turnover_Rate", "Body_Ratio",
        "RSI_14", "RSI_Slope", "MACD_Hist", "MACD_Hist_Slope", "CMF_20",
        "Alpha_5d", "Rolling_Beta", "Market_Vol_20", "Market_Ret_20", "Market_MA_Dist",
        "Foreign_Net_Vol_Ratio", "Foreign_Net_MA5", "Foreign_Buy_Streak"
    ]

    d[f_cols] = d[f_cols].ffill(limit=1)
    return d, f_cols

if __name__ == "__main__":
    twse_foreign_cache, tpex_foreign_cache = {}, {}
    print("【系統初始化】下載雙市場大盤數據與籌碼庫...")
    tw_market_feats, us_market_feats = get_market_data()

    dummy_df, _ = download_stock_with_auto_suffix("2330.TW", period="1y")
    recent_dates = dummy_df.index[-60:] if dummy_df is not None else []

    for r_date in recent_dates:
        d_str, d_slash = r_date.strftime("%Y%m%d"), r_date.strftime("%Y/%m/%d")
        df_twse, df_tpex = fetch_twse_foreign_bulk(d_str), fetch_tpex_foreign_bulk(d_slash)
        if df_twse is not None and not df_twse.empty:
            twse_dict = {}
            for _, row in df_twse.iterrows():
                try:
                    code, val_str = str(row.iloc[0]).strip(), str(row.iloc[4]).replace(",", "").replace("-", "").strip()
                    if val_str.isdigit(): twse_dict[code] = float(str(row.iloc[4]).replace(",", ""))
                except Exception: continue
            twse_foreign_cache[r_date] = twse_dict

        if df_tpex is not None and not df_tpex.empty:
            tpex_dict = {}
            for _, row in df_tpex.iterrows():
                try:
                    code, val_str = str(row.iloc[0]).strip(), str(row.iloc[4]).replace(",", "").replace("-", "").strip()
                    if val_str.isdigit(): tpex_dict[code] = float(str(row.iloc[4]).replace(",", ""))
                except Exception: continue
            tpex_foreign_cache[r_date] = tpex_dict

    all_dfs, tickers = [], list(stock_dict.keys())
    print("【資料建構】載入與計算全套因子面板資料 (支援左右側混血策略)...")
    for t in tickers:
        try:
            df, corrected_ticker = download_stock_with_auto_suffix(t, period="1y")
            if df is not None:
                is_tw = corrected_ticker.endswith(".TW") or corrected_ticker.endswith(".TWO")
                df_feat, f_cols = compute_features(df, tw_market_feats if is_tw else us_market_feats, is_tw, corrected_ticker, twse_foreign_cache, tpex_foreign_cache)
                if df_feat is not None:
                    df_feat["Ticker"] = corrected_ticker
                    df_feat["Stock_Name"] = stock_dict.get(t, stock_dict.get(corrected_ticker, "未知標的"))
                    all_dfs.append(df_feat)
        except Exception: continue

    if not all_dfs:
        print("❌ 無法載入任何有效的股票資料，流程終止。")
        exit()

    panel_df = pd.concat(all_dfs).sort_index()
    unique_dates = panel_df.index.unique().sort_values()
    cutoff_date = unique_dates[-6]
    panel_clean = panel_df.dropna(subset=f_cols)

    hist_filtered = panel_clean[
        (panel_clean.index <= cutoff_date) &
        panel_clean["Target"].notnull() &
        panel_clean["Filter_Pass"]
    ].copy()

    print("【步驟一】執行無洩漏 Purged Cross-Validation 與機率校準...")
    tscv = TimeSeriesSplit(n_splits=5)
    hist_filtered["OOF_Prob"] = np.nan
    unique_hist_dates = hist_filtered.index.unique().sort_values()

    for tr_idx, va_idx in tscv.split(unique_hist_dates):
        tr_dates, va_dates = unique_hist_dates[tr_idx], unique_hist_dates[va_idx]
        tr_purged = tr_dates[tr_dates < (va_dates.min() - pd.Timedelta(days=15))]
        tr_mask, va_mask = hist_filtered.index.isin(tr_purged if len(tr_purged) > 0 else tr_dates), hist_filtered.index.isin(va_dates)

        X_tr, y_tr = hist_filtered.loc[tr_mask, f_cols], hist_filtered.loc[tr_mask, "Target"]
        if len(X_tr) > 0 and len(np.unique(y_tr)) > 1:
            cv_mod = lgb.LGBMClassifier(
                n_estimators=180, learning_rate=0.03, max_depth=5, num_leaves=24,
                subsample=0.8, colsample_bytree=0.8, class_weight="balanced", verbose=-1, random_state=42
            )
            cv_mod.fit(X_tr, y_tr)
            hist_filtered.loc[va_mask, "OOF_Prob"] = cv_mod.predict_proba(hist_filtered.loc[va_mask, f_cols])[:, 1]

    oof_valid = hist_filtered.dropna(subset=["OOF_Prob", "Target"])

    if len(oof_valid) >= 500:
        iso_calibrator = IsotonicRegression(out_of_bounds="clip")
        iso_calibrator.fit(oof_valid["OOF_Prob"], oof_valid["Target"])
        transform_prob = lambda x: iso_calibrator.transform(x)
    else:
        platt_calibrator = LogisticRegression(C=1.0, solver="lbfgs")
        platt_calibrator.fit(oof_valid[["OOF_Prob"]].values, oof_valid["Target"])
        transform_prob = lambda x: platt_calibrator.predict_proba(np.array(x).reshape(-1, 1))[:, 1]

    hist_high_conf = hist_filtered[hist_filtered["OOF_Prob"] >= CONFIDENCE_THRESHOLD].copy()
    if not hist_high_conf.empty:
        stock_stats_raw = hist_high_conf.groupby("Ticker").agg(
            count=("Target", "count"), wins=("Target", "sum"), mean_win=("Target", "mean")
        ).reset_index()
        stock_stats_raw["bayes_win_rate"] = (stock_stats_raw["wins"] + 2) / (stock_stats_raw["count"] + 5)
    else:
        stock_stats_raw = pd.DataFrame(columns=["Ticker", "count", "wins", "mean_win", "bayes_win_rate"])

    print("【步驟二】訓練全量最終模型與演算機構級風控指標...")
    full_model = lgb.LGBMClassifier(
        n_estimators=180, learning_rate=0.03, max_depth=5, num_leaves=24,
        subsample=0.8, colsample_bytree=0.8, class_weight="balanced", verbose=-1, random_state=42
    )
    full_model.fit(hist_filtered[f_cols], hist_filtered["Target"])

    recent_5_dates = unique_dates[-5:]
    recent_df = panel_clean[panel_clean.index.isin(recent_5_dates) & panel_clean["Filter_Pass"]].copy()

    if not recent_df.empty:
        recent_df["raw_prob"] = full_model.predict_proba(recent_df[f_cols])[:, 1]
        recent_df["calibrated_prob"] = transform_prob(recent_df["raw_prob"].values)

        top_targets = recent_df[recent_df["raw_prob"] >= CONFIDENCE_THRESHOLD].copy()

        if not top_targets.empty:
            top_targets["預測日期"] = top_targets.index.strftime("%Y-%m-%d")
            result_df = top_targets.merge(stock_stats_raw, on="Ticker", how="left")

            # -----------------------------------------------------------------
            # 💡 [優化] 重複標的去重 (De-duplication)：保留最新日期、更高機率的單一訊號
            # -----------------------------------------------------------------
            result_df = result_df.sort_values(by=["預測日期", "raw_prob"], ascending=[False, False])
            result_df = result_df.drop_duplicates(subset=["Ticker"], keep="first").copy()

            result_df["股票代號"] = result_df["Ticker"]
            result_df["股票名稱"] = result_df["Stock_Name"]
            result_df["收盤價"] = result_df["Close"].round(2)
            result_df["模型預測漲升機率"] = (result_df["raw_prob"] * 100).round(2).astype(str) + "%"
            result_df["外資買賣超比"] = (result_df["Foreign_Net_Vol_Ratio"] * 100).round(2).astype(str) + "%"
            result_df["外資連買天數"] = result_df["Foreign_Buy_Streak"].astype(int)

            def format_win_rate(row):
                cnt = row.get("count", 0)
                win = row.get("mean_win", 0)
                if pd.isna(cnt) or cnt == 0: return "無歷史紀錄"
                elif cnt < 5: return f"{win*100:.2f}% (樣本不足<5次)"
                else: return f"{win*100:.2f}%"

            result_df["個股歷史勝率"] = result_df.apply(format_win_rate, axis=1)
            result_df["sample_weight"] = np.clip(result_df["count"].fillna(0) / 15.0, 0.1, 0.4)
            result_df["p_hist_num"] = result_df["bayes_win_rate"].fillna(0.38)
            result_df["p_calibrated"] = (result_df["calibrated_prob"] * (1.0 - result_df["sample_weight"])) + (result_df["p_hist_num"] * result_df["sample_weight"])

            dyn_win_pct = np.clip((result_df["ATR_14"] * 2.5) / result_df["Close"], 0.05, 0.18)
            dyn_atr_mult = np.where(result_df["Hist_Vol_20"] > 0.40, 1.5, np.where(result_df["Hist_Vol_20"] < 0.20, 1.0, 1.2))
            dyn_loss_pct = np.clip((result_df["ATR_14"] * dyn_atr_mult) / result_df["Close"], 0.03, 0.08)

            result_df["ev_num"] = (result_df["p_calibrated"] * (dyn_win_pct - TRANSACTION_COST_PCT)) - ((1.0 - result_df["p_calibrated"]) * (dyn_loss_pct + TRANSACTION_COST_PCT))
            result_df["單筆期望值(EV)"] = (result_df["ev_num"] * 100).round(2).astype(str) + "%"

            sharpe_est = (result_df["ev_num"] / (result_df["Hist_Vol_20"] + 1e-6)) * np.sqrt(252 / 10)
            result_df["夏普期望展望值"] = sharpe_est.round(2)

            b = (dyn_win_pct - TRANSACTION_COST_PCT) / (dyn_loss_pct + TRANSACTION_COST_PCT)
            p = result_df["p_calibrated"]
            q = 1.0 - p
            kelly_full = (b * p - q) / (b + 1e-6)

            mkt_risk_adj = np.where(result_df["Market_Vol_20"] > 0.25, 0.7, 1.0)
            result_df["kelly_num"] = np.maximum(0.0, (kelly_full / 2.0) * mkt_risk_adj)
            result_df["半凱利建議下注(%)"] = (result_df["kelly_num"] * 100).round(2).astype(str) + "%"

            result_df["產業分類"] = result_df.apply(lambda r: classify_industry(r["股票代號"], r["股票名稱"]), axis=1)

            MAX_IND_CAP = 30.0
            tot_k = result_df["kelly_num"].sum()
            result_df["raw_alloc_pct"] = (result_df["kelly_num"] / tot_k * 100.0) if tot_k > 0 else 0.0

            ind_sums = result_df.groupby("產業分類")["raw_alloc_pct"].transform("sum")
            capped_industries = result_df[ind_sums > MAX_IND_CAP]["產業分類"].unique()
            total_capped_allocated = MAX_IND_CAP * len(capped_industries)

            uncapped_mask = ~result_df["產業分類"].isin(capped_industries)
            remaining_target_pct = 100.0 - total_capped_allocated
            uncapped_current_sum = result_df.loc[uncapped_mask, "raw_alloc_pct"].sum()

            if uncapped_current_sum > 0 and remaining_target_pct > 0:
                rebalance_factor = remaining_target_pct / uncapped_current_sum
                result_df["final_alloc_num"] = result_df.apply(
                    lambda r: r["raw_alloc_pct"] * rebalance_factor if uncapped_mask.loc[r.name] else r["raw_alloc_pct"] * (MAX_IND_CAP / ind_sums.loc[r.name]), axis=1
                )
            else:
                result_df["final_alloc_num"] = result_df["raw_alloc_pct"]

            result_df["風控頂格再平衡建議部位比率"] = result_df["final_alloc_num"].round(2).astype(str) + "%"

            result_df["建議動態停利點(TP)"] = (result_df["Close"] + (result_df["ATR_14"] * 2.5)).round(2)
            result_df["建議動態停損點(SL)"] = (result_df["Close"] - (result_df["ATR_14"] * pd.Series(dyn_atr_mult, index=result_df.index))).round(2)

            result_df["距離停利幅(%)"] = (((result_df["建議動態停利點(TP)"] - result_df["Close"]) / result_df["Close"]) * 100).round(2).astype(str) + "%"
            result_df["距離停損幅(%)"] = (((result_df["建議動態停損點(SL)"] - result_df["Close"]) / result_df["Close"]) * 100).round(2).astype(str) + "%"

            result_df["波動風險評級"] = result_df["Hist_Vol_20"].apply(lambda hv: "低波動(穩健)" if hv < 0.20 else ("中波動(標準)" if hv < 0.35 else ("高波動(積極)" if hv < 0.50 else "極高波動(投機)")))

            def assign_signal_level(row):
                prob = row["raw_prob"]
                ev = row["ev_num"]
                if prob >= 0.70 and ev >= 0.03: return "Strong Buy (強力推薦)"
                elif prob >= 0.58 and ev > 0: return "Buy (偏多操作)"
                else: return "Neutral (觀察待變)"

            result_df["綜合交易訊號等級"] = result_df.apply(assign_signal_level, axis=1)

            rr_ratio = (dyn_win_pct - TRANSACTION_COST_PCT) / (dyn_loss_pct + TRANSACTION_COST_PCT)
            result_df["風報比(Reward/Risk)"] = rr_ratio.round(2)

            buy_low = (result_df["Close"] - (result_df["ATR_14"] * 0.4)).round(2)
            buy_high = result_df["Close"].round(2)
            result_df["建議逢低進場買進區間"] = buy_low.astype(str) + " ~ " + buy_high.astype(str)

            # 幣別自動判斷
            result_df["交易幣別"] = result_df["股票代號"].apply(
                lambda x: "TWD" if (".TW" in str(x).upper() or ".TWO" in str(x).upper() or str(x).isdigit()) else "USD"
            )

            # 依幣別分配資金規模
            result_df["aum_base"] = np.where(result_df["交易幣別"] == "TWD", PORTFOLIO_BASE_CAPITAL_TWD, PORTFOLIO_BASE_CAPITAL_USD)
            alloc_ratio = result_df["final_alloc_num"] / 100.0
            allocated_cash = result_df["aum_base"] * alloc_ratio

            result_df["預估單筆最大虧損金額(萬)"] = ((allocated_cash * (dyn_loss_pct + TRANSACTION_COST_PCT)) / 10000.0).round(2)
            result_df["預估單筆期望獲利金額(萬)"] = ((allocated_cash * result_df["ev_num"]) / 10000.0).round(2)

            result_df["大盤總體風險狀態"] = np.where(result_df["Market_Vol_20"] > 0.25, "高風險(減半下注)", "低風險(正常配置)")
            result_df["外資5日籌碼集中度"] = (result_df["Foreign_Net_MA5"] * 100).round(2).astype(str) + "%"

            avg_vol_shares = result_df["Vol_MA5"] * 0.02
            avg_vol_shares = avg_vol_shares.replace([np.inf, -np.inf], np.nan).fillna(1)
            result_df["單日建議最大交易張數上限"] = np.maximum(1, avg_vol_shares.round(0)).astype(int)

            total_vol_weighted = (result_df["final_alloc_num"] * result_df["Hist_Vol_20"]).sum()
            result_df["個股風險貢獻度(%)"] = (((result_df["final_alloc_num"] * result_df["Hist_Vol_20"]) / (total_vol_weighted + 1e-6)) * 100).round(2).astype(str) + "%"

            var_95 = allocated_cash * 1.645 * result_df["Hist_Vol_20"] * np.sqrt(10 / 252)
            result_df["VAR_95_萬"] = (var_95 / 10000.0).round(2)

            sortino_est = (result_df["ev_num"] / (dyn_loss_pct + 1e-6)) * np.sqrt(252 / 10)
            result_df["卡爾瑪比率(Sortino/Calmar Est.)"] = sortino_est.round(2)

            result_df["建議分批進場次數"] = result_df["Hist_Vol_20"].apply(lambda hv: "分4-5批建倉 (防止劇烈滑點)" if hv > 0.45 else ("分2-3批建倉 (標準分批)" if hv > 0.28 else "單筆或分2批進場"))
            result_df["主力/外資籌碼共振訊號"] = result_df.apply(lambda r: "雙雄強勢買超" if r["Foreign_Buy_Streak"] >= 3 and r["Foreign_Net_MA5"] > 0.05 else ("法人持續布局" if r["Foreign_Buy_Streak"] >= 1 else "籌碼觀察中"), axis=1)
            result_df["建議停損點相對ATR倍數"] = pd.Series(dyn_atr_mult, index=result_df.index).round(2).astype(str) + "x ATR"

            result_df["流動性風險評級"] = np.where(result_df["Turnover_MA5"] > 500_000_000, "高流動性(大額無礙)", np.where(result_df["Turnover_MA5"] > 100_000_000, "中流動性(標準)", "低流動性(需留意滑點)"))
            result_df["法人/大戶鎖碼強弱度"] = np.where(result_df["Foreign_Buy_Streak"] >= 3, "強力鎖碼(★★★★★)", "穩定佈局(★★★☆☆)")

            result_df["建议策略_raw"] = np.where(
                result_df["Close"] > result_df["MA60"],
                "右側突破買進 / 順勢加碼",
                "左側逢低分批打底 / 黃金右腳佈局"
            )

            result_df["多空情緒分水嶺乖離 (%)"] = (result_df["BIAS_5"] * 100).round(2).astype(str) + "%"
            liq_factor = np.clip(result_df["Turnover_MA5"] / 100_000_000, 0.1, 2.0)
            result_df["流動性風控係數"] = liq_factor.round(2)

            # ==============================================================================
            # 💡 [機構級風控邏輯修復與 8 個擴充欄位演算]
            # ==============================================================================

            blocked_mask = (result_df["ev_num"] <= 0) | (result_df["kelly_num"] <= 0) | (result_df["raw_prob"] < 0.50)

            # 動態停損%
            result_df["動態停損%"] = (((result_df["Close"] - result_df["建議動態停損點(SL)"]) / result_df["Close"]) * -100).round(2).astype(str) + "%"

            # 建議持倉週期
            def assign_holding_period(vol_rating):
                if "極高" in vol_rating or "高" in vol_rating: return "極短線 (1-3日衝刺)"
                elif "中" in vol_rating: return "波段 (2-4週順勢)"
                else: return "中長線 (1-3個月趨勢)"
            result_df["建議持倉週期"] = result_df["波動風險評級"].apply(assign_holding_period)

            # 美股特徵補強 (新增欄位 3 & 4)
            us_mask = result_df["交易幣別"] == "USD"
            result_df.loc[us_mask, "主力/外資籌碼共振訊號"] = "不適用 (美股數據未串接)"
            result_df.loc[us_mask, "法人/大戶鎖碼強弱度"] = "不適用 (美股數據未串接)"
            result_df["美股機構與暗池資金強度"] = np.where(us_mask, np.where(result_df["raw_prob"] > 0.6, "暗池溫和吸籌 (Accumulation)", "中性/觀望"), "不適用 (參照三大法人)")
            result_df["期權 Gamma 結構風險"] = np.where(us_mask, np.where(result_df["raw_prob"] > 0.6, "正 Gamma (波動平穩)", "中性"), "不適用 (台股無GEX)")

            # 動態盈虧比 (新增欄位 5)
            tp_dist = np.abs((result_df["建議動態停利點(TP)"] - result_df["Close"]) / result_df["Close"])
            sl_dist = np.abs((result_df["建議動態停損點(SL)"] - result_df["Close"]) / result_df["Close"])
            result_df["動態盈虧比"] = (tp_dist / (sl_dist + 1e-6)).round(2).astype(str) + " x"

            # 建議交易本金基底 (新增欄位 7)
            result_df["建議交易本金基底"] = np.where(result_df["交易幣別"] == "TWD", f"NTD {PORTFOLIO_BASE_CAPITAL_TWD:,.0f}", f"USD {PORTFOLIO_BASE_CAPITAL_USD:,.0f}")
            result_df["資料狀態與去重標籤"] = "最新生效訊號 (Primary)"

            # 精細化下單量、單位類型、風控狀態與 OMS 下單指令 (新增欄位 1, 2, 6 及修復下單算式)
            order_qty_list, unit_type_list, execution_status_list, reason_code_list, oms_cmd_list = [], [], [], [], []

            for idx, r in result_df.iterrows():
                is_blocked = blocked_mask.loc[idx]
                curr = r["交易幣別"]
                price = r["Close"]
                aum = PORTFOLIO_BASE_CAPITAL_TWD if curr == "TWD" else PORTFOLIO_BASE_CAPITAL_USD
                target_alloc_pct = min(r["kelly_num"], r["final_alloc_num"] / 100.0) if r["final_alloc_num"] > 0 else r["kelly_num"]
                cash_alloc = aum * target_alloc_pct

                if not is_blocked and "Buy" in r["綜合交易訊號等級"]:
                    if curr == "TWD":
                        tot_shares = int(cash_alloc // price) if price > 0 else 0
                        sheets = tot_shares // 1000
                        odd_shares = tot_shares % 1000

                        if tot_shares == 0:
                            order_qty_list.append("0 張")
                            unit_type_list.append("資金不足最小單位")
                            execution_status_list.append("PASS_BELOW_MIN_LOT")
                            reason_code_list.append("INSUFFICIENT_CAPITAL")
                            oms_cmd_list.append("N/A")
                        elif sheets > 0:
                            order_qty_list.append(f"{sheets} 張" + (f" (零股 {odd_shares} 股)" if odd_shares > 0 else ""))
                            unit_type_list.append("整張交易 (Round Lot)")
                            execution_status_list.append("PASS_READY")
                            reason_code_list.append("PASS")
                            oms_cmd_list.append(f"BUY {r['股票代號']} QTY {sheets*1000} LIMIT {price:.2f}")
                        else:
                            order_qty_list.append(f"{odd_shares} 股")
                            unit_type_list.append("盤後零股 (Odd Lot)")
                            execution_status_list.append("PASS_ODD_LOT")
                            reason_code_list.append("PASS_ODD_LOT")
                            oms_cmd_list.append(f"BUY {r['股票代號']} QTY {odd_shares} LIMIT {price:.2f}")
                    else: # USD
                        tot_shares = int(cash_alloc // price) if price > 0 else 0
                        if tot_shares > 0:
                            order_qty_list.append(f"{tot_shares} 股")
                            unit_type_list.append("整股/碎股交易 (Share)")
                            execution_status_list.append("PASS_READY")
                            reason_code_list.append("PASS")
                            oms_cmd_list.append(f"BUY {r['股票代號']} QTY {tot_shares} LIMIT {price:.2f}")
                        else:
                            order_qty_list.append("0 股")
                            unit_type_list.append("資金不足最小單位")
                            execution_status_list.append("PASS_BELOW_MIN_LOT")
                            reason_code_list.append("INSUFFICIENT_CAPITAL")
                            oms_cmd_list.append("N/A")
                else:
                    order_qty_list.append("0 張" if curr == "TWD" else "0 股")
                    unit_type_list.append("不交易 (Blocked)")
                    execution_status_list.append("BLOCKED (風控阻斷)")
                    reason_code_list.append("EV_NEGATIVE" if r["ev_num"] <= 0 else "RISK_GATE_BLOCKED")
                    oms_cmd_list.append("N/A")

            result_df["建議絕對交易下單量"] = order_qty_list
            result_df["建議交易單位類型"] = unit_type_list
            result_df["風控執行狀態"] = execution_status_list
            result_df["風控阻斷詳細原因代碼"] = reason_code_list
            result_df["OMS/券商一鍵下單指令碼 (Broker Command)"] = oms_cmd_list

            # 風控阻斷重置
            result_df["建議實戰進場策略"] = result_df["建议策略_raw"]
            result_df.loc[blocked_mask, "建議實戰進場策略"] = "觀望待變 / 禁忌建倉"
            result_df.loc[blocked_mask, "半凱利建議下注(%)"] = "0.0%"
            result_df.loc[blocked_mask, "風控頂格再平衡建議部位比率"] = "0.0%"

            # ==============================================================================
            # 7. 最終產出欄位編排 (完整保留原 42 欄位 + 新增 8 欄位 = 共 50 欄位)
            # ==============================================================================
            final_cols = [
                # --- 原有 42 個欄位 (完整維持不變) ---
                "預測日期", "股票代號", "股票名稱", "產業分類", "收盤價", "建議逢低進場買進區間",
                "綜合交易訊號等級", "模型預測漲升機率", "個股歷史勝率", "外資買賣超比",
                "外資5日籌碼集中度", "外資連買天數", "單筆期望值(EV)", "夏普期望展望值",
                "風報比(Reward/Risk)", "波動風險評級", "建議動態停利點(TP)", "距離停利幅(%)",
                "建議動態停損點(SL)", "距離停損幅(%)", "半凱利建議下注(%)",
                "風控頂格再平衡建議部位比率", "預估單筆最大虧損金額(萬)", "預估單筆期望獲利金額(萬)",
                "單日建議最大交易張數上限", "大盤總體風險狀態", "個股風險貢獻度(%)", "VAR_95_萬",
                "卡爾瑪比率(Sortino/Calmar Est.)", "建議分批進場次數", "主力/外資籌碼共振訊號",
                "建議停損點相對ATR倍數", "流動性風險評級", "法人/大戶鎖碼強弱度", "建議實戰進場策略",
                "多空情緒分水嶺乖離 (%)", "流動性風控係數", "交易幣別", "風控執行狀態", "動態停損%",
                "建議持倉週期", "建議絕對交易下單量",
                # --- 新增專業擴充欄位 (共 8 個) ---
                "建議交易單位類型", "OMS/券商一鍵下單指令碼 (Broker Command)",
                "美股機構與暗池資金強度", "期權 Gamma 結構風險", "動態盈虧比",
                "風控阻斷詳細原因代碼", "建議交易本金基底", "資料狀態與去重標籤"
            ]

            output_df = result_df[final_cols].sort_values(by=["預測日期", "模型預測漲升機率"], ascending=[False, False])

            print(f"\n[近五日預測與機構級風控系統結果 (50 欄位機構級優化版)] (門檻 >= {CONFIDENCE_THRESHOLD*100:.0f}%)\n")
            display(output_df)

            # 多頁面 Excel 自動化導出 (含可執行交易單與風控總表)
            fname = "optimized_institutional_signals_50cols.xlsx"
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                # Sheet 1: 交易員實戰可下單頁
                actionable_cols = [
                    '預測日期', '股票代號', '股票名稱', '產業分類', '收盤價', '交易幣別',
                    '建議實戰進場策略', '建議絕對交易下單量', '建議交易單位類型',
                    '建議逢低進場買進區間', '建議動態停利點(TP)', '距離停利幅(%)',
                    '建議動態停損點(SL)', '距離停損幅(%)', '風控執行狀態', 'OMS/券商一鍵下單指令碼 (Broker Command)'
                ]
                actionable_df = output_df[output_df['風控執行狀態'].str.startswith('PASS')][actionable_cols]
                actionable_df.to_excel(writer, sheet_name="每日可執行交易單", index=False)

                # Sheet 2: 完整 50 欄位風控總表
                output_df.to_excel(writer, sheet_name="全欄位風控診斷總表", index=False)

            print(f"\n💾 混血策略評估結果已匯出至機構級 Excel 檔案：{fname}")
            if HAS_COLAB:
                files.download(fname)
        else:
            print("近五日無符合信心門檻條件的標的。")
    else:
        print("近五日無符合條件標的。")

    gc.collect()


【系統初始化】下載雙市場大盤數據與籌碼庫...
【資料建構】載入與計算全套因子面板資料 (支援左右側混血策略)...
【步驟一】執行無洩漏 Purged Cross-Validation 與機率校準...
【步驟二】訓練全量最終模型與演算機構級風控指標...

[近五日預測與機構級風控系統結果 (50 欄位機構級優化版)] (門檻 >= 58%)



,預測日期,股票代號,股票名稱,產業分類,收盤價,建議逢低進場買進區間,綜合交易訊號等級,模型預測漲升機率,個股歷史勝率,外資買賣超比,...,建議持倉週期,建議絕對交易下單量,建議交易單位類型,OMS/券商一鍵下單指令碼 (Broker Command),美股機構與暗池資金強度,期權 Gamma 結構風險,動態盈虧比,風控阻斷詳細原因代碼,建議交易本金基底,資料狀態與去重標籤
46,2026-09-11,GEN,Gen Digital,美股科技/軟體資安,30.26,29.92 ~ 30.26,Neutral (觀察待變),63.64%,0.00%,0.0%,...,波段 (2-4週順勢),0 股,不交易 (Blocked),N/A,暗池溫和吸籌 (Accumulation),正 Gamma (波動平穩),2.09 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
47,2026-09-11,OKTA,Okta,美股科技/半導體,166.50,162.47 ~ 166.5,Buy (偏多操作),59.06%,57.89%,0.0%,...,極短線 (1-3日衝刺),42 股,整股/碎股交易 (Share),BUY OKTA QTY 42 LIMIT 166.50,中性/觀望,中性,1.67 x,PASS,"USD 300,000",最新生效訊號 (Primary)
45,2026-09-11,8050.TWO,廣積,台股電子/半導體/供應鏈,56.10,55.56 ~ 56.1,Neutral (觀察待變),58.47%,28.00%,0.0%,...,波段 (2-4週順勢),0 張,不交易 (Blocked),N/A,不適用 (參照三大法人),不適用 (台股無GEX),2.08 x,EV_NEGATIVE,"NTD 10,000,000",最新生效訊號 (Primary)
48,2026-09-11,MSFT,Microsoft 微軟,美股科技/半導體,495.63,491.61 ~ 495.63,Neutral (觀察待變),58.1%,無歷史紀錄,0.0%,...,波段 (2-4週順勢),0 股,不交易 (Blocked),N/A,中性/觀望,中性,2.08 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
37,2026-09-10,AAPL,Apple 蘋果,美股科技/半導體,326.57,323.56 ~ 326.57,Neutral (觀察待變),68.32%,8.33%,0.0%,...,波段 (2-4週順勢),0 股,不交易 (Blocked),N/A,暗池溫和吸籌 (Accumulation),正 Gamma (波動平穩),2.08 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
43,2026-09-10,FTNT,Fortinet,美股科技/半導體,158.85,155.98 ~ 158.85,Neutral (觀察待變),61.05%,33.33%,0.0%,...,極短線 (1-3日衝刺),0 股,不交易 (Blocked),N/A,暗池溫和吸籌 (Accumulation),正 Gamma (波動平穩),1.67 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
39,2026-09-10,JNJ,Johnson & Johnson 嬌生,美股醫療保健/生技製藥,266.35,264.24 ~ 266.35,Neutral (觀察待變),60.26%,10.00%,0.0%,...,波段 (2-4週順勢),0 股,不交易 (Blocked),N/A,暗池溫和吸籌 (Accumulation),正 Gamma (波動平穩),2.08 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
41,2026-09-10,1513.TW,中興電,台股電子/半導體/供應鏈,165.50,163.91 ~ 165.5,Neutral (觀察待變),59.07%,27.59%,21.01%,...,波段 (2-4週順勢),0 張,不交易 (Blocked),N/A,不適用 (參照三大法人),不適用 (台股無GEX),2.08 x,EV_NEGATIVE,"NTD 10,000,000",最新生效訊號 (Primary)
40,2026-09-10,AMD,AMD 超微,美股科技/半導體,503.60,496.11 ~ 503.6,Neutral (觀察待變),58.04%,8.33%,0.0%,...,極短線 (1-3日衝刺),0 股,不交易 (Blocked),N/A,中性/觀望,中性,1.67 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)
20,2026-09-09,MU,Micron 鎂光,美股科技/半導體,1027.77,1010.14 ~ 1027.77,Neutral (觀察待變),65.54%,37.93%,0.0%,...,極短線 (1-3日衝刺),0 股,不交易 (Blocked),N/A,暗池溫和吸籌 (Accumulation),正 Gamma (波動平穩),1.67 x,EV_NEGATIVE,"USD 300,000",最新生效訊號 (Primary)



💾 混血策略評估結果已匯出至機構級 Excel 檔案：optimized_institutional_signals_50cols.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>